<a href="https://colab.research.google.com/github/simonlim563/asl-ml-immersion/blob/master/custom_ai_cross_video/clients/VN_Zott/optimization_zott_vn_21082025_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# def prepare_notebook():

#     """    This function pulls the .py files from cloud storage bucket and installs the necessary dependancies    """
#     import os # Create a directory to store the copied files
#     from datetime import datetime, timedelta
#     os.makedirs('executor_files', exist_ok=True)# Use gsutil to copy the .py files

#     # Generate yesterday's date string (YYYYMMDD)
#     yesterday_str = (datetime.today() - timedelta(days=1)).strftime("%Y%m%d")

#     # Build the GCS path for yesterday's historical adjustments file
#     historical_file_gcs = f"gs://apac-github-test/Files_for_vertex_ai/Cross_Video_Zott_historical/historical_adjustments_{yesterday_str}.parquet"

#     !gsutil -m cp gs://apac-github-test/Files_for_vertex_ai/mp-adh-xaxis-apac-eb039826de5b.json executor_files/
#     !gsutil -m cp gs://apac-github-test/Files_for_vertex_ai/mp-adh-groupm-sg-cea49b7167eb.json executor_files/
#     !gsutil -m cp gs://apac-github-test/Files_for_vertex_ai/*.py executor_files/

#     !gsutil -m cp gs://apac-github-test/Files_for_vertex_ai/Cross_Video_Zott_historical/historical_adjustments_20250812.parquet executor_files/

#     # Add the directory to your Python path
#     import sys
#     sys.path.append('executor_files')

#     !pip install requests
#     !pip install pandas
#     !pip install numpy
#     !pip install scipy
#     !pip install six
#     !pip install google-cloud-secret-manager
#     !pip install google-cloud-bigquery
#     !pip install google-api-python-client
#     !pip install google-auth
#     !pip install google-auth-httplib2
#     !pip install google-api-core
#     !pip install google-cloud-storage
#     !pip install db-dtypes

#     import MS_DV360API_Connector

#     return print("Everything has been installed successfully - notebook good to go!")

# prepare_notebook()

In [2]:
import json
import pandas as pd
from datetime import date, timedelta, datetime
import numpy as np
from scipy.optimize import minimize
import requests
from contextlib import closing
from six.moves.urllib.request import urlopen
from googleapiclient import discovery
from io import StringIO
from googleapiclient.discovery import build
from google.oauth2 import service_account
from googleapiclient import http as googleHttp
import time
import random
import os
import db_dtypes
from MS_DV360API_Connector import DV360Connector
from urllib.parse import urlencode
from six.moves.urllib.parse import urlencode, urlunparse
from six import string_types
import io
import datetime as dt
from google.cloud import storage
from google.cloud import bigquery
from decimal import Decimal


# import seaborn as sns
# import matplotlib.pyplot as plt

### Load DV360 report

In [3]:
current_date = datetime.now()

# Set up the parameters to request an API report

json_file_path = "executor_files/mp-adh-groupm-sg-cea49b7167eb.json"

if not os.path.exists(json_file_path):
    print(f"Error: JSON file not found at {json_file_path}")
    exit(1)

try:
    with open(json_file_path, 'r', encoding='utf-8') as f:
        config_data = json.load(f)
    print(config_data)

except FileNotFoundError:
    print(f"Error: JSON file not found at {json_file_path}")
    exit(1)

except json.JSONDecodeError as e:
    print(f"Error decoding JSON: {e}")
    exit(1)

except Exception as e:
    print(f"An unexpected error occurred while loading JSON: {e}")
    exit(1)

dv360_connector = DV360Connector('mp-adh-groupm-sg-cea49b7167eb.json')
io_id = ['1022529463', '1022173941']    ## This is actually IO ID
start_date = '2025-07-31'   ## The report start date
end_date = '2025-08-31'

print(f"The report will include activity from {start_date} to {end_date}")

## Below is the list of categorical columns and metrics the report will include (divided by columns and metrics)
## You can find more elements and its Console equivalent here https://developers.google.com/bid-manager/v1.1/filters-metrics

columns_list = ['FILTER_DATE',
                'FILTER_PARTNER_NAME',
                'FILTER_PARTNER',
                'FILTER_COUNTRY',
                 'FILTER_ADVERTISER_CURRENCY',
                'FILTER_ADVERTISER_NAME',
                 'FILTER_ADVERTISER',
                'FILTER_INSERTION_ORDER_NAME',
                 'FILTER_INSERTION_ORDER',
                'FILTER_LINE_ITEM_NAME',
                 'FILTER_LINE_ITEM',
                #  'FILTER_DEVICE_TYPE',
                # 'FILTER_PAGE_LAYOUT'
                 ]

metrics_list = ['METRIC_IMPRESSIONS','METRIC_TRUEVIEW_VIEWS', 'METRIC_RICH_MEDIA_VIDEO_PLAYS', 'METRIC_RICH_MEDIA_VIDEO_COMPLETIONS','METRIC_TOTAL_MEDIA_COST_ADVERTISER','METRIC_REVENUE_ADVERTISER']

## CODE TO RUN THE API REPORT BELOW

report_definition = dv360_connector.build_report_definition(metrics_list, io_id, columns_list, start_date, end_date)
report_raw = dv360_connector.download_report(report_definition)


Error: JSON file not found at executor_files/mp-adh-groupm-sg-cea49b7167eb.json
Error: JSON file not found at executor_files/mp-adh-groupm-sg-cea49b7167eb.json
The report will include activity from 2025-07-31 to 2025-08-31
('1507673956',) ('5255280240',) QUEUED
('1507673956',) ('5255280240',) QUEUED
('1507673956',) ('5255280240',) DONE


In [4]:
report_df = report_raw.iloc[:-3]

In [5]:
report_df.tail()

,Date,Partner,Partner ID,Country,Advertiser Currency,Advertiser,Advertiser ID,Insertion Order,Insertion Order ID,Line Item,Line Item ID,Impressions,TrueView: Views,Starts (Video),Complete Views (Video),Total Media Cost (Advertiser Currency),Revenue (Adv Currency)
158,2025/08/21,NMS Vietnam,1278635.0,VN,VND,{WM} {BU:VN} Zott_VND,6.993274e+09,XVN25-0295 Zott Yogurt Launching Jun to Sep'25...,1.022174e+09,Instream only_Fe18-34_G2_Computer,2.285288e+10,1282,0,1143,914,28110.922367,28110.922367
159,2025/08/21,NMS Vietnam,1278635.0,VN,VND,{WM} {BU:VN} Zott_VND,6.993274e+09,XVN25-0295 Zott Yogurt Launching Jun to Sep'25...,1.022174e+09,Instream only_Fe18-34_G2_Smartphone,2.285360e+10,2314,0,2228,1710,30849.890444,30849.890444
160,2025/08/21,NMS Vietnam,1278635.0,VN,VND,{WM} {BU:VN} Zott_VND,6.993274e+09,XVN25-0295 Zott Yogurt Launching Jun to Sep'25...,1.022174e+09,Instream only_Fe18-34_G2_Tablet,2.285360e+10,123,0,116,78,2740.733205,2740.733205
161,2025/08/21,NMS Vietnam,1278635.0,VN,VND,{WM} {BU:VN} Zott_VND,6.993274e+09,XVN25-0295 Zott Yogurt Launching Jun to Sep'25...,1.022529e+09,YT_Category Buyer_Fe18-34_G2_CTV_new,2.287171e+10,4364,0,4406,3977,98886.669504,98886.669504
162,2025/08/21,NMS Vietnam,1278635.0,VN,VND,{WM} {BU:VN} Zott_VND,6.993274e+09,XVN25-0295 Zott Yogurt Launching Jun to Sep'25...,1.022529e+09,YT_Category Buyer_Fe18-34_G2_Computer_new,2.287552e+10,1774,0,1802,1719,61362.366944,61362.366944


### Load TikTok report

In [6]:
ACCESS_TOKEN = "d118cb774b86a7eae6a5a06a1164e8573efa264c"

In [7]:
ADVERTISER_ID = [7371003546684424193]  # INPUT THE ADVERTISER ID directly from Tiktok
START_DATE = date(2025, 7, 31)
END_DATE = date(2025, 8, 31)

In [8]:
# Set the metrics you want to pull from the report

SERVICE_TYPE = "AUCTION"
REPORT_TYPE = "BASIC"
DATA_LEVEL = "AUCTION_AD"
DIMENSIONS = ["ad_id", "stat_time_day"]
METRICS = [
    #Basic data metrics
    "impressions",
    "clicks",
    "total_landing_page_view",
    "conversion",
    "result",
    "secondary_goal_result",
    "spend",

    # Video play metrics
     "video_play_actions",
    # "video_watched_2s",
    # "video_watched_6s",
    # "average_video_play",
    # "video_views_p25",
    # "video_views_p50",
    # "video_views_p75",
    # "video_views_p100",
    # "engaged_view",
     "engaged_view_15s",


    # Basic data metrics - Reach
    # "reach",
    # "frequency",

    #Engagement metrics
    # "engagements",
    # "profile_visits",
    # "likes",
    # "comments",
    # "shares",
    # "follows",
    # "anchor_clicks",

    #ROAS metrics
    # "onsite_shopping",
    # "cost_per_onsite_shopping",
    # "onsite_shopping_rate",
    # "value_per_onsite_shopping",
    # "total_onsite_shopping_value",



    # attributes
    "currency",
    "campaign_id",
    "adgroup_id",
    "campaign_budget",
    "objective_type",
    "budget",
    "billing_event",
    "bid_strategy",
    "bid",
    #"split_test"
]


PAGE_SIZE = 1000
QUERY_MODE = 'CHUNK'

In [9]:
# API Query for pulling the dimension keys and metrics

PATH = "/open_api/v1.3/report/task/create/"


def build_url(path, query=""):
    # type: (str, str) -> str
    """
    Build request URL
    :param path: Request path
    :param query: Querystring
    :return: Request URL
    """
    scheme, netloc = "https", "business-api.tiktok.com"
    return urlunparse((scheme, netloc, path, "", query, ""))


def post(json_str):
    # type: (str) -> dict
    """
    Send GET request
    :param json_str: Args in JSON format
    :return: Response in JSON format
    """
    args = json.loads(json_str)
    query_string = urlencode(
        {
            k: v if isinstance(v, string_types) else json.dumps(v)
            for k, v in args.items()
        }
    )
    url = build_url(PATH, query_string)
    headers = {"Access-Token": ACCESS_TOKEN}
    rsp = requests.post(url, headers=headers)
    return rsp.json()


def batch_iterator(ids, batch_size):
    for i in range(0, len(ids), batch_size):
        yield ids[i : i + batch_size]


if __name__ == "__main__":

    service_type = SERVICE_TYPE
    report_type = REPORT_TYPE
    data_level = DATA_LEVEL
    dimensions_list = DIMENSIONS
    dimensions = json.dumps(dimensions_list)
    metrics_list = METRICS
    metrics = json.dumps(metrics_list)
    start_date = START_DATE.strftime("%Y-%m-%d")
    end_date = END_DATE.strftime("%Y-%m-%d")
    page_size = PAGE_SIZE
    advertiser_task_map = {}
    task_unsuccessful = {}

    # query_mode = QUERY_MODE

    data_df = pd.DataFrame()
    # Args in JSON format

    for index, advertiser_id in enumerate(ADVERTISER_ID):
        # my_args = "{\"metrics\": %s, \"data_level\": \"%s\", \"end_date\": \"%s\", \"page_size\": \"%s\", \"start_date\": \"%s\", \"advertiser_id\": \"%s\",  \"report_type\": \"%s\", \"dimensions\": %s, \"service_type\": \"%s\" }" % (metrics, data_level, end_date, page_size, start_date, advertiser_id, report_type, dimensions, service_type)
        my_args = (
            '{"advertiser_id": "%s", "service_type": "%s", "report_type": "%s",  "data_level": "%s", "dimensions": %s, "metrics": %s, "start_date": "%s", "end_date": "%s", "page_size": %s, "enable_report_title_translation": false }'
            % (
                advertiser_id,
                service_type,
                report_type,
                data_level,
                dimensions,
                metrics,
                start_date,
                end_date,
                page_size,
            )
        )

        result = post(my_args)
        #print(result)
        if result["code"] == 0:
            task_id = result["data"]["task_id"]
            advertiser_task_map[advertiser_id] = task_id
            # print(advertiser_task_map)
        else:
            message = result["message"]
            task_unsuccessful[advertiser_id] = message

        if index > 0 and index % 9 == 0:
            time.sleep(15)
        else:
            time.sleep(3)

    # for advertiser_id, message in task_unsuccessful:
    #     print(f"task creation was unsuccessful for {advertiser_id} due to {message}")




In [10]:
time.sleep(300)

In [11]:


PATH_CHECK = "/open_api/v1.3/report/task/check/"
PATH_DOWNLOAD = "/open_api/v1.3/report/task/download/"


def build_url(path, query=""):
    # type: (str, str) -> str
    """
    Build request URL
    :param path: Request path
    :param query: Querystring
    :return: Request URL
    """
    scheme, netloc = "https", "business-api.tiktok.com"
    return urlunparse((scheme, netloc, path, "", query, ""))


def check(json_str):
    # type: (str) -> dict
    """
    Send GET request
    :param json_str: Args in JSON format
    :return: Response in JSON format
    """
    args = json.loads(json_str)
    query_string = urlencode(
        {
            k: v if isinstance(v, string_types) else json.dumps(v)
            for k, v in args.items()
        }
    )
    url = build_url(PATH_CHECK, query_string)
    headers = {
        "Access-Token": ACCESS_TOKEN,
    }
    rsp = requests.get(url, headers=headers)
    return rsp.json()


def download(json_str):
    # type: (str) -> dict
    """
    Send GET request
    :param json_str: Args in JSON format
    :return: Response in JSON format
    """
    args = json.loads(json_str)
    query_string = urlencode(
        {
            k: v if isinstance(v, string_types) else json.dumps(v)
            for k, v in args.items()
        }
    )
    url = build_url(PATH_DOWNLOAD, query_string)
    headers = {
        "Access-Token": ACCESS_TOKEN,
    }
    rsp = requests.get(url, headers=headers)
    return rsp.text


if __name__ == "__main__":
    load_df = pd.DataFrame()
    not_ready = {}

    for advertiser_id, task_id in advertiser_task_map.items():

        # Args in JSON format
        my_args = '{"advertiser_id": "%s", "task_id": "%s"}' % (advertiser_id, task_id)
        check_task = check(my_args)
        #print(check_task)

        if check_task["code"] == 0 and check_task["data"]["status"] == "SUCCESS":

            csv_string = download(my_args)
            temp_df = pd.read_csv(io.StringIO(csv_string))
            temp_df["Advertiser ID"] = advertiser_id
            if not temp_df.empty:
                load_df = pd.concat([temp_df, load_df])

        else:
            print(check_task)
            # print(
            #     "The taskid: {} for advertiser {} is not ready yet".format(
            #         task_id, advertiser_id
            #     )
            # )
            # not_ready[advertiser_id] = task_id

        time.sleep(5)

In [12]:
load_df.tail()

,campaign_id,adgroup_id,ad_id,stat_time_day,impressions,clicks,total_landing_page_view,conversion,result,secondary_goal_result,...,video_play_actions,engaged_view_15s,currency,campaign_budget,objective_type,budget,billing_event,bid_strategy,bid,Advertiser ID
897,1838977845599410,1838977911053554,1838978951501953,2025-07-31,57690,124,52,0,55355,-,...,57342,1876,VND,-,Reach,126159804.0,Guaranteed Delivery(GD),Standard Bid,-,7371003546684424193
898,1839054926886305,1839054926886321,1839788951446545,2025-08-07,690,3,1,0,59,-,...,683,59,VND,99000000.0,Video View,656375.0,CPV,Maximum delivery,-,7371003546684424193
899,1834709069183346,1834549067161922,1836086486675521,2025-08-15,0,0,0,0,0,-,...,0,0,VND,-,Reach,385274468.0,Guaranteed Delivery(GD),Standard Bid,-,7371003546684424193
900,1839054926886305,1839054926886321,1839054926889473,2025-08-18,469,2,0,0,64,-,...,464,64,VND,99000000.0,Video View,656375.0,CPV,Maximum delivery,-,7371003546684424193
901,1838970473891890,1838970615975457,1838971791943810,2025-08-02,65490,127,37,0,62613,-,...,65167,1353,VND,-,Reach,145640797.0,Guaranteed Delivery(GD),Standard Bid,-,7371003546684424193


In [13]:
# prompt: from load_df, filter for rows that have 1839054926886305,1839054793361409 as campaign_id. reset index

filtered_df = load_df[load_df['campaign_id'].isin([1839054926886305, 1839054793361409])]
filtered_df = filtered_df.reset_index(drop=True)
filtered_df.head()


,campaign_id,adgroup_id,ad_id,stat_time_day,impressions,clicks,total_landing_page_view,conversion,result,secondary_goal_result,...,video_play_actions,engaged_view_15s,currency,campaign_budget,objective_type,budget,billing_event,bid_strategy,bid,Advertiser ID
0,1839054793361409,1839054793361425,1839155866521042,2025-08-16,11723,31,17,0,1320,-,...,11632,1320,VND,99000000.0,Video View,498363.0,CPV,Maximum delivery,-,7371003546684424193
1,1839054926886305,1839058353004833,1839788951446577,2025-08-08,2274,18,1,0,368,-,...,2179,368,VND,99000000.0,Video View,963695.0,CPV,Maximum delivery,-,7371003546684424193
2,1839054926886305,1839058353004833,1839058353005761,2025-08-07,760,3,1,0,131,-,...,750,131,VND,99000000.0,Video View,963695.0,CPV,Maximum delivery,-,7371003546684424193
3,1839054793361409,1839054793361425,1839054793362449,2025-08-05,1731,9,5,0,253,-,...,1719,253,VND,99000000.0,Video View,498363.0,CPV,Maximum delivery,-,7371003546684424193
4,1839054926886305,1839054926886321,1839054926888241,2025-08-21,86,0,0,0,6,-,...,86,6,VND,99000000.0,Video View,656375.0,CPV,Maximum delivery,-,7371003546684424193


In [14]:
pd.set_option('display.float_format', lambda x: f'{x:.6f}' if abs(x) < 1e6 else f'{x:.0f}')

### Import historical data

In [20]:
def clean_adgroup_id(series):
  series = series.astype(str).str.strip()
  series = series.str.replace(r'\.0$', '', regex=True)
  series = series.apply(lambda x: int(float(x)))
  return series.astype('int64')

In [23]:
yesterday_str = (datetime.today() - timedelta(days=1)).strftime("%Y%m%d")

# Read CSV
historical_df = pd.read_parquet(f"historical_adjustments_{yesterday_str}.parquet")

# Drop NaNs
historical_df = historical_df.dropna(subset=['adgroup_id'])

# Clean adgroup_id
historical_df['adgroup_id'] = clean_adgroup_id(historical_df['adgroup_id'])

# Parse date safely
historical_df['date'] = pd.to_datetime(historical_df['date'], errors='coerce')

# Force entire column to be datetime64[ns] to avoid mixed types
historical_df['date'] = pd.to_datetime(historical_df['date'])

# Rename for consistency
historical_df = historical_df.rename(columns={'platform_total_budget': 'total_budget'})

# Debug check
print(historical_df['date'].dtype)  # should be datetime64[ns]
print(historical_df['date'].unique())

datetime64[ns]
<DatetimeArray>
['2025-08-08 00:00:00', '2025-08-09 00:00:00', '2025-08-10 00:00:00',
 '2025-08-11 00:00:00', '2025-08-12 00:00:00', '2025-08-13 00:00:00',
 '2025-08-14 00:00:00', '2025-08-15 00:00:00', '2025-08-16 00:00:00',
 '2025-08-17 00:00:00', '2025-08-18 00:00:00', '2025-08-19 00:00:00',
 '2025-08-20 00:00:00']
Length: 13, dtype: datetime64[ns]


In [24]:
# # Read CSV
# historical_df = pd.read_csv("historical_13080205 - Sheet1.csv", dtype={'adgroup_id': str})

# # Drop NaNs
# historical_df = historical_df.dropna(subset=['adgroup_id'])

# # Clean adgroup_id
# historical_df['adgroup_id'] = clean_adgroup_id(historical_df['adgroup_id'])

# # Parse date safely
# historical_df['date'] = pd.to_datetime(historical_df['date'], errors='coerce')

# # Fix misinterpreted 2025-09-08 → 2025-08-09
# def fix_sept8_to_aug9(d):
#   if pd.notna(d) and isinstance(d, pd.Timestamp) and d.month == 9 and d.day == 8:
#       return pd.Timestamp(year=d.year, month=8, day=9)
#   return d

# historical_df['date'] = historical_df['date'].apply(fix_sept8_to_aug9)

# # Force entire column to be datetime64[ns] to avoid mixed types
# historical_df['date'] = pd.to_datetime(historical_df['date'])

# # Rename for consistency
# historical_df = historical_df.rename(columns={'platform_total_budget': 'total_budget'})

# # Debug check
# print(historical_df['date'].dtype)  # should be datetime64[ns]
# print(historical_df['date'].unique())

In [1]:
# # Save Parquet (safe for precision)
# historical_df.to_parquet('historical_adjustments_20250813.parquet', index=False)

# # Save CSV (for sharing, with float precision control)
# historical_df.to_csv('historical_adjustments_20250813.csv', index=False, float_format="%.15g")

In [25]:
historical_df.tail(11)

,adgroup_id,combined_index,spend_guardlines_max,spend_guardlines_min,new_budget,allocated_budget,total_budget,date
132,22852881341,1.039638,262230.721041,87410.240347,262230.721040,474528.408312,736759.129352,2025-08-20
133,22853602793,1.158655,249147.659363,83049.219788,167283.417995,474734.947316,642018.365311,2025-08-20
134,22853603024,1.081361,28517.061022,9505.687007,28517.061020,98880.274371,127397.335392,2025-08-20
135,22871714300,1.052774,340464.897202,113488.299067,283617.193233,553080.286151,836697.479384,2025-08-20
136,22871714861,1.072997,316486.782498,105495.594166,253766.401715,528619.816551,782386.218266,2025-08-20
137,22875518767,1.080545,47061.643565,15687.214522,47061.643564,123889.413630,170951.057194,2025-08-20
138,22875518857,1.013606,189659.848290,63219.949430,189659.848290,400609.553668,590269.401958,2025-08-20
139,1839054793361425,1.362442,142563.295672,47521.098557,80094.675805,419640.217569,499734.893374,2025-08-20
140,1839054926886321,1.189033,180845.843139,60281.947713,146978.936662,501721.274677,648700.211339,2025-08-20
141,1839057950747698,1.109628,345238.023810,115079.341270,210134.766655,784972.342431,995107.109086,2025-08-20


In [26]:
historical_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143 entries, 0 to 142
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   adgroup_id            143 non-null    int64         
 1   combined_index        143 non-null    float64       
 2   spend_guardlines_max  143 non-null    float64       
 3   spend_guardlines_min  143 non-null    float64       
 4   new_budget            143 non-null    float64       
 5   allocated_budget      143 non-null    float64       
 6   total_budget          143 non-null    float64       
 7   date                  143 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(6), int64(1)
memory usage: 9.1 KB


### Manual analysis of performance

#### DV360

In [15]:
# prompt: group report_df by 'Insertion Order ID'. Keep only 'Insertion Order', 'Insertion Order ID', 'Impressions', 'Starts (Video)', 'Complete Views (Video)' and 'Revenue (Adv Currency)' and store it as a different df. Calculate new metric vcr which is calculated by dividing completed views by starts. Also calculate cpm

# Keep only specified columns
report_df_grouped = report_df[[
    'Insertion Order ID',
    'Insertion Order',
    'Impressions',
    'Starts (Video)',
    'Complete Views (Video)',
    'Revenue (Adv Currency)'
]].copy()

# Group by 'Insertion Order ID' and sum relevant metrics
report_df_grouped = report_df_grouped.groupby('Insertion Order ID').sum().reset_index()

# Calculate VCR
report_df_grouped['VCR'] = (
    report_df_grouped['Complete Views (Video)'] /
    report_df_grouped['Impressions']
).fillna(0)  # Handle potential division by zero

# Calculate CPM
report_df_grouped['CPM'] = (
    report_df_grouped['Revenue (Adv Currency)'] /
    report_df_grouped['Impressions'] * 1000
).fillna(0) # Handle potential division by zero

report_df_grouped


,Insertion Order ID,Insertion Order,Impressions,Starts (Video),Complete Views (Video),Revenue (Adv Currency),VCR,CPM
0,1022173941,XVN25-0295 Zott Yogurt Launching Jun to Sep'25...,1425504,1423240,1181375,20012657,0.828742,14039.004426
1,1022529463,XVN25-0295 Zott Yogurt Launching Jun to Sep'25...,1073673,1072190,955885,31525913,0.890294,29362.676220


In [16]:
# Mappings the old line items into the new one
id_mapping = {
 22853599586: 22871714861,  # CTV
 22852881533: 22875518857,  # Computer
 22852881101: 22871714300,  # Smartphone
 22857485470: 22875518767   # Tablet
}

name_mapping = {
 'YT_Category Buyer_Fe18-34_G2_CTV': 'YT_Category Buyer_Fe18-34_G2_CTV_new',
 'YT_Category Buyer_Fe18-34_G2_Computer': 'YT_Category Buyer_Fe18-34_G2_Computer_new',
 'YT_Category Buyer_Fe18-34_G2_Smartphone': 'YT_Category Buyer_Fe18-34_G2_Smartphone_new',
 'YT_Category Buyer_Fe18-34_G2_Tablet': 'YT_Category Buyer_Fe18-34_G2_Tablet_new'
}

# Replace line_item_id and line_item_name using the mappings
report_df['Line Item ID'] = report_df['Line Item ID'].replace(id_mapping)
report_df['Line Item'] = report_df['Line Item'].replace(name_mapping)

/tmp/ipython-input-2315877081.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  report_df['Line Item ID'] = report_df['Line Item ID'].replace(id_mapping)
/tmp/ipython-input-2315877081.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  report_df['Line Item'] = report_df['Line Item'].replace(name_mapping)


In [17]:
# prompt: group report_df by 'Line Item ID'. Keep only 'Insertion Order ID', 'Line Item ID', 'Line Item', 'Impressions', 'Starts (Video)', 'Complete Views (Video)' and 'Revenue (Adv Currency)' and store it as a different df. Calculate new metric vcr which is calculated by dividing completed views by starts. Also calculate cpm

# Keep only specified columns and group by 'Line Item ID'
report_df_grouped = report_df[[
    'Insertion Order ID',
    'Line Item ID',
    'Line Item',
    'Impressions',
    'Starts (Video)',
    'Complete Views (Video)',
    'Revenue (Adv Currency)'
]].copy()

report_df_grouped = report_df_grouped.groupby('Line Item ID').sum().reset_index()

# Calculate VCR
report_df_grouped['VCR'] = (
    report_df_grouped['Complete Views (Video)'] /
    report_df_grouped['Impressions']
).fillna(0)  # Handle potential division by zero

# Calculate CPM
report_df_grouped['CPM'] = (
    report_df_grouped['Revenue (Adv Currency)'] /
    report_df_grouped['Impressions'] * 1000
).fillna(0) # Handle potential division by zero

report_df_grouped

,Line Item ID,Insertion Order ID,Line Item,Impressions,Starts (Video),Complete Views (Video),Revenue (Adv Currency),VCR,CPM
0,22852881341,22487826702,Instream only_Fe18-34_G2_ComputerInstream only...,543366,542409,478981,8845293,0.881507,16278.701413
1,22853602793,22487826702,Instream only_Fe18-34_G2_SmartphoneInstream on...,747596,746452,590703,8479338,0.790137,11342.138982
2,22853603024,22487826702,Instream only_Fe18-34_G2_TabletInstream only_F...,134377,134216,111610,2684337,0.830574,19976.161695
3,22857490753,19421304879,Instream only_Fe18-34_G2_CTVInstream only_Fe18...,165,163,81,3689.679469,0.490909,22361.693752
4,22871714300,19428059797,YT_Category Buyer_Fe18-34_G2_Smartphone_newYT_...,288211,286684,257617,10282986,0.893849,35678.674115
5,22871714861,20450589260,YT_Category Buyer_Fe18-34_G2_CTV_newYT_Categor...,481527,481091,422276,10036701,0.876952,20843.486083
6,22875518767,19428059797,YT_Category Buyer_Fe18-34_G2_Tablet_newYT_Cate...,87612,87336,75681,3243281,0.863820,37018.682446
7,22875518857,20450589260,YT_Category Buyer_Fe18-34_G2_Computer_newYT_Ca...,216323,217079,200311,7962944,0.925981,36810.437125


In [18]:
#dv360 report
report_df.head(1)

,Date,Partner,Partner ID,Country,Advertiser Currency,Advertiser,Advertiser ID,Insertion Order,Insertion Order ID,Line Item,Line Item ID,Impressions,TrueView: Views,Starts (Video),Complete Views (Video),Total Media Cost (Advertiser Currency),Revenue (Adv Currency)
0,2025/07/31,NMS Vietnam,1278635,VN,VND,{WM} {BU:VN} Zott_VND,6993273759,XVN25-0295 Zott Yogurt Launching Jun to Sep'25...,1022173941,Instream only_Fe18-34_G2_Computer,22852881341,8824,0,8810,7858,169601.906020,169601.906020


In [19]:
#tiktok report
filtered_df.head(1)

,campaign_id,adgroup_id,ad_id,stat_time_day,impressions,clicks,total_landing_page_view,conversion,result,secondary_goal_result,...,video_play_actions,engaged_view_15s,currency,campaign_budget,objective_type,budget,billing_event,bid_strategy,bid,Advertiser ID
0,1839054793361409,1839054793361425,1839155866521042,2025-08-16,11723,31,17,0,1320,-,...,11632,1320,VND,99000000.0,Video View,498363.000000,CPV,Maximum delivery,-,7371003546684424193


### Push data into bigquery

#### BQ for DV360 REPORT

In [ ]:
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account
from google.api_core.exceptions import NotFound, Conflict
from decimal import Decimal

In [ ]:
# -----------------------
# CONFIGURATION
# -----------------------
SERVICE_ACCOUNT_FILE = "mp-adh-xaxis-apac-eb039826de5b.json"
PROJECT_ID = "mp-adh-xaxis-apac"
DATASET_ID = "custom_ai_solution"
TABLE_ID = "vn_zott_cross_video_aug_sept_DV360"
LOCATION = "asia-southeast1"

In [ ]:
# Authenticate
credentials = service_account.Credentials.from_service_account_file(
SERVICE_ACCOUNT_FILE,
scopes=[
    "https://www.googleapis.com/auth/bigquery",
    "https://www.googleapis.com/auth/drive",
    "https://www.googleapis.com/auth/spreadsheets",
],
)
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

In [ ]:
# -----------------------
# STEP 1: CLEAN COLUMN NAMES
# -----------------------
def clean_column_name(col: str) -> str:
  col = col.lower()
  col = col.replace(":", "_").replace("(", "").replace(")", "")
  col = col.replace(" ", "_").replace("__", "_")
  return col

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
  df = df.copy()
  df.columns = [clean_column_name(c) for c in df.columns]

  # Convert date column
  if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date

  # Convert IDs to nullable integers
  id_columns = ["partner_id", "advertiser_id", "insertion_order_id", "line_item_id"]
  for col in id_columns:
    if col in df.columns:
      df[col] = df[col].astype("Int64")

  # Convert currency columns to Decimal for BigQuery NUMERIC
  currency_columns = ["total_media_cost_advertiser_currency", "revenue_adv_currency"]
  for col in currency_columns:
    if col in df.columns:
      df[col] = df[col].apply(lambda x: Decimal(str(x)) if pd.notna(x) else None)

  return df

In [ ]:
# -----------------------
# STEP 2: CREATE DATASET IF NOT EXISTS
# -----------------------
def create_dataset_if_not_exists(dataset_id: str):
  dataset_ref = f"{PROJECT_ID}.{dataset_id}"
  try:
      client.get_dataset(dataset_ref)
      print(f"✅ Dataset {dataset_id} already exists.")
  except NotFound:
      dataset = bigquery.Dataset(dataset_ref)
      dataset.location = LOCATION  # Set Singapore location
      dataset = client.create_dataset(dataset, exists_ok=True)
      print(f"✅ Created dataset {dataset_id} in {LOCATION}.")


In [ ]:
# -----------------------
# STEP 3: CREATE TABLE IF NOT EXISTS
# -----------------------
def create_table_if_not_exists(dataset_id: str, table_id: str, schema: list):
  table_ref = f"{PROJECT_ID}.{dataset_id}.{table_id}"
  try:
      client.get_table(table_ref)
      print(f"Table {table_id} already exists.")
  except NotFound:
      table = bigquery.Table(table_ref, schema=schema)
      client.create_table(table)
      print(f"Created table {table_id}.")

In [ ]:
# -----------------------
# STEP 4: LOAD DATA (TRUNCATE)
# -----------------------
def load_dataframe_to_bq(df: pd.DataFrame, dataset_id: str, table_id: str, schema: list):
  table_ref = f"{PROJECT_ID}.{dataset_id}.{table_id}"
  job_config = bigquery.LoadJobConfig(
      schema=schema,
      write_disposition="WRITE_TRUNCATE"
  )
  job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
  job.result()
  print(f"Loaded {len(df)} rows into {table_ref} (old data truncated).")

In [ ]:
# -----------------------
# SCHEMA DEFINITION
# -----------------------
campaign_info_schema = [
bigquery.SchemaField("date", "DATE", mode="REQUIRED"),
bigquery.SchemaField("partner", "STRING", mode="REQUIRED"),
bigquery.SchemaField("partner_id", "INT64", mode="REQUIRED"),
bigquery.SchemaField("country", "STRING", mode="REQUIRED"),
bigquery.SchemaField("advertiser_currency", "STRING", mode="REQUIRED"),
bigquery.SchemaField("advertiser", "STRING", mode="REQUIRED"),
bigquery.SchemaField("advertiser_id", "INT64", mode="REQUIRED"),
bigquery.SchemaField("insertion_order", "STRING", mode="REQUIRED"),
bigquery.SchemaField("insertion_order_id", "INT64", mode="REQUIRED"),
bigquery.SchemaField("line_item", "STRING", mode="REQUIRED"),
bigquery.SchemaField("line_item_id", "INT64", mode="REQUIRED"),
bigquery.SchemaField("impressions", "INT64", mode="REQUIRED"),
bigquery.SchemaField("trueview_views", "INT64", mode="REQUIRED"),
bigquery.SchemaField("starts_video", "INT64", mode="REQUIRED"),
bigquery.SchemaField("complete_views_video", "INT64", mode="REQUIRED"),
bigquery.SchemaField("total_media_cost_advertiser_currency", "NUMERIC", mode="REQUIRED"),
bigquery.SchemaField("revenue_adv_currency", "NUMERIC", mode="REQUIRED")
]

In [ ]:
# -----------------------
# MAIN EXECUTION
# -----------------------
# Example: Replace with your actual DataFrame
# report_df = pd.read_csv("your_file.csv")
report_df_1 = clean_dataframe(report_df)

# Create dataset
create_dataset_if_not_exists(DATASET_ID)

# Create table
create_table_if_not_exists(DATASET_ID, TABLE_ID, campaign_info_schema)

# Load data
load_dataframe_to_bq(report_df_1, DATASET_ID, TABLE_ID, campaign_info_schema)

print("DV360 REPORT UPDATED ✅")

✅ Dataset custom_ai_solution already exists.
Table vn_zott_cross_video_aug_sept_DV360 already exists.
Loaded 108 rows into mp-adh-xaxis-apac.custom_ai_solution.vn_zott_cross_video_aug_sept_DV360 (old data truncated).
DV360 REPORT UPDATED ✅


#### BQ for Tiktok report

In [ ]:
# -----------------------
# CONFIGURATION
# -----------------------
SERVICE_ACCOUNT_FILE = "mp-adh-xaxis-apac-eb039826de5b.json"
PROJECT_ID = "mp-adh-xaxis-apac"
DATASET_ID = "custom_ai_solution"
TABLE_ID_TIKTOK = "vn_zott_cross_video_aug_sept_tiktok"
LOCATION = "asia-southeast1"  # Singapore

In [ ]:
# Authenticate
credentials = service_account.Credentials.from_service_account_file(
  SERVICE_ACCOUNT_FILE,
  scopes=[
      "https://www.googleapis.com/auth/bigquery",
      "https://www.googleapis.com/auth/drive",
      "https://www.googleapis.com/auth/spreadsheets",
  ],
)
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

In [ ]:
# -----------------------
# STEP 1: CLEAN DATAFRAME
# -----------------------
def clean_column_name(col: str) -> str:
  col = col.strip().lower().replace(" ", "_").replace(":", "_").replace("(", "").replace(")", "")
  col = col.replace("__", "_")
  return col

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
  df = df.copy()
  df.columns = [clean_column_name(c) for c in df.columns]

  # Convert date column
  if "stat_time_day" in df.columns:
      df["stat_time_day"] = pd.to_datetime(df["stat_time_day"], errors="coerce").dt.date

  # Convert numeric IDs to nullable integers
  id_columns = ["campaign_id", "adgroup_id", "ad_id", "advertiser_id"]
  for col in id_columns:
      if col in df.columns:
          df[col] = df[col].astype("Int64")

  # Convert spend/budget to Decimal for NUMERIC
  currency_columns = ["spend", "budget"]
  for col in currency_columns:
      if col in df.columns:
          df[col] = df[col].apply(lambda x: Decimal(str(x)) if pd.notna(x) else None)

  return df

In [ ]:
# -----------------------
# STEP 2: CREATE DATASET IF NOT EXISTS
# -----------------------
def create_dataset_if_not_exists(dataset_id: str):
  dataset_ref = f"{PROJECT_ID}.{dataset_id}"
  try:
      client.get_dataset(dataset_ref)
      print(f"✅ Dataset {dataset_id} already exists.")
  except NotFound:
      dataset = bigquery.Dataset(dataset_ref)
      dataset.location = LOCATION
      client.create_dataset(dataset, exists_ok=True)
      print(f"✅ Created dataset {dataset_id} in {LOCATION}.")

In [ ]:
# -----------------------
# STEP 3: CREATE TABLE IF NOT EXISTS
# -----------------------
def create_table_if_not_exists(dataset_id: str, table_id: str, schema: list):
  table_ref = f"{PROJECT_ID}.{dataset_id}.{table_id}"
  try:
      client.get_table(table_ref)
      print(f"✅ Table {table_id} already exists.")
  except NotFound:
      table = bigquery.Table(table_ref, schema=schema)
      client.create_table(table)
      print(f"✅ Created table {table_id} in dataset {dataset_id}.")

In [ ]:
# -----------------------
# STEP 4: LOAD DATA (TRUNCATE)
# -----------------------
def load_dataframe_to_bq(df: pd.DataFrame, dataset_id: str, table_id: str, schema: list):
  table_ref = f"{PROJECT_ID}.{dataset_id}.{table_id}"
  job_config = bigquery.LoadJobConfig(
      schema=schema,
      write_disposition="WRITE_TRUNCATE"
  )
  job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
  job.result()
  print(f"✅ Loaded {len(df)} rows into {table_ref} (old data truncated).")

In [ ]:
# -----------------------
# SCHEMA FOR TIKTOK DATA
# -----------------------
tiktok_schema = [
  bigquery.SchemaField("campaign_id", "INT64", mode="REQUIRED"),
  bigquery.SchemaField("adgroup_id", "INT64", mode="REQUIRED"),
  bigquery.SchemaField("ad_id", "INT64", mode="REQUIRED"),
  bigquery.SchemaField("stat_time_day", "DATE", mode="REQUIRED"),
  bigquery.SchemaField("impressions", "INT64", mode="REQUIRED"),
  bigquery.SchemaField("clicks", "INT64", mode="REQUIRED"),
  bigquery.SchemaField("total_landing_page_view", "INT64", mode="REQUIRED"),
  bigquery.SchemaField("conversion", "INT64", mode="REQUIRED"),
  bigquery.SchemaField("result", "INT64", mode="REQUIRED"),
  bigquery.SchemaField("secondary_goal_result", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("spend", "NUMERIC", mode="REQUIRED"),
  bigquery.SchemaField("video_play_actions", "INT64", mode="REQUIRED"),
  bigquery.SchemaField("engaged_view_15s", "INT64", mode="REQUIRED"),
  bigquery.SchemaField("currency", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("campaign_budget", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("objective_type", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("budget", "NUMERIC", mode="REQUIRED"),
  bigquery.SchemaField("billing_event", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("bid_strategy", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("bid", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("advertiser_id", "INT64", mode="REQUIRED")
]


In [ ]:
# -----------------------
# MAIN EXECUTION
# -----------------------
# Example: Replace with your actual TikTok DataFrame
# tiktok_df = pd.read_csv("tiktok_data.csv")
tiktok_df_clean = clean_dataframe(filtered_df)

create_dataset_if_not_exists(DATASET_ID)
create_table_if_not_exists(DATASET_ID, TABLE_ID_TIKTOK, tiktok_schema)
load_dataframe_to_bq(tiktok_df_clean, DATASET_ID, TABLE_ID_TIKTOK, tiktok_schema)

print("✅ TikTok report uploaded to BigQuery")

✅ Dataset custom_ai_solution already exists.
✅ Created table vn_zott_cross_video_aug_sept_tiktok in dataset custom_ai_solution.
✅ Loaded 278 rows into mp-adh-xaxis-apac.custom_ai_solution.vn_zott_cross_video_aug_sept_tiktok (old data truncated).
✅ TikTok report uploaded to BigQuery


### Initialising platform ids

In [27]:
#initialise the platform ids
dv_adgroups = report_df['Line Item ID'].unique().astype(int).tolist()
tiktok_adgroups = filtered_df['adgroup_id'].unique().astype(int).tolist()
tiktok_adgroup_kol = [1839057950747698,1839054793361425]
tiktok_adgroup_nuclass= [1839058353004833,1839054926886321]
dv_adgroup_id_video = [22853603024,22852881341,22853602793]
dv_adgroup_id_yt = [22871714861,22875518857,22871714300,22875518767]
dv_adgroups_1 = dv_adgroup_id_yt +dv_adgroup_id_video
print(dv_adgroups)
print(tiktok_adgroups)
print(dv_adgroups_1)

[22852881341, 22853602793, 22853603024, 22871714861, 22875518857, 22871714300, 22875518767, 22857490753]
[1839054793361425, 1839058353004833, 1839054926886321, 1839057950747698]
[22871714861, 22875518857, 22871714300, 22875518767, 22853603024, 22852881341, 22853602793]


### Initialising campaign details

In [28]:
Overall_total_budget_dv = 198000000
Overall_total_budget_tt = 198000000
total_base_budget_tt = 138600000
total_base_budget_dv =  148500000
dv_base_yt_budget = 148500000 * (120000000/(120000000 + 78000000))
dv_base_video_budget = 148500000 * (78000000/(120000000 + 78000000))
floating_budget = Overall_total_budget_dv + Overall_total_budget_tt - total_base_budget_dv - total_base_budget_tt
print('Total overall floating budget :', floating_budget)
print('DV360 YT base budget :',dv_base_yt_budget)
print('DV360 Video base budget:',dv_base_video_budget)
Combined_budget = Overall_total_budget_dv + Overall_total_budget_tt
print(dv_base_yt_budget)

Total overall floating budget : 108900000
DV360 YT base budget : 90000000.0
DV360 Video base budget: 58500000.0
90000000.0


In [29]:
Percentage_TikTok_base_budget = total_base_budget_tt/Overall_total_budget_tt
Percentage_DV360_base_budget = total_base_budget_dv/Overall_total_budget_dv
print('Percentage of base budget for TIKTOK',Percentage_TikTok_base_budget)
print('Percentage of base budget for DV360',Percentage_DV360_base_budget)

Percentage of base budget for TIKTOK 0.7
Percentage of base budget for DV360 0.75


In [30]:
Total_budget = floating_budget
Start_date = dt.date(2025,7,31)
End_date = dt.date(2025,9,30)
today = date.today()

### Data processing to split float and spend budget

In [88]:
#Converting date data types
report_df.loc[:, 'Date'] = pd.to_datetime(report_df['Date']).dt.date
filtered_df.loc[:, 'stat_time_day'] = pd.to_datetime(filtered_df['stat_time_day']).dt.date

In [89]:
# Define the split logic for after 1st budget allocation
def split_budget(row):
  if pd.notna(row['total_budget']) and row['spend'] >= 0.90 * row['total_budget']:
      base = row['spend'] * (row['allocated_budget'] / row['total_budget'])
      flt = row['spend'] * (row['new_budget'] / row['total_budget'])
      rule = "Proportional"
  else:
      base = row['spend'] * 0.75
      flt = row['spend'] * 0.25
      rule = "Fixed 75/25"
  return pd.Series([base, flt, rule])

In [90]:
# Define the split logic to be more strict
def split_budget_new(row):
  if pd.notna(row['total_budget']) and row['spend'] >= 0.95 * row['total_budget']:
      base = row['spend'] * (row['allocated_budget'] / row['total_budget'])
      flt = row['spend'] * (row['new_budget'] / row['total_budget'])
      rule = "Proportional"
  else:
      base = row['spend'] * 0.75
      flt = row['spend'] * 0.25
      rule = "Fixed 75/25"
  return pd.Series([base, flt, rule])

#### TikTok split

In [91]:
# Filter for learning/initial phase dates to do 70/30 split

tt_data = filtered_df.loc[(filtered_df['stat_time_day'] >= Start_date) & (filtered_df['stat_time_day'] < date(2025, 8, 8))]
tt_data = tt_data.rename(columns={'stat_time_day': 'Date'})
tt_data = tt_data.loc[tt_data['adgroup_id'].isin(tiktok_adgroups),['adgroup_id','Date','impressions','engaged_view_15s','spend']]
tt_data = tt_data.groupby(by=['Date','adgroup_id']).sum().reset_index()
tt_data = tt_data[~((tt_data['impressions'] == 0) & (tt_data['spend'] == 0))].reset_index(drop=True)
tt_data['base_spend'] = tt_data['spend'] * 0.7
tt_data['float_spend'] = tt_data['spend'] * 0.3
tt_data.tail()

,Date,adgroup_id,impressions,engaged_view_15s,spend,base_spend,float_spend
27,2025-08-06,1839058353004833,30521,3050,868591.000000,608013.700000,260577.300000
28,2025-08-07,1839054793361425,36782,4395,867489.000000,607242.300000,260246.700000
29,2025-08-07,1839054926886321,26295,2695,864876.000000,605413.200000,259462.800000
30,2025-08-07,1839057950747698,42971,7256,870638.000000,609446.600000,261191.400000
31,2025-08-07,1839058353004833,23698,2675,862587.000000,603810.900000,258776.100000


In [92]:
# Filter for dates after the 1st change which will follow a new logic split.

tt_data_post = filtered_df.loc[(filtered_df['stat_time_day'] > date(2025, 8, 7)) & (filtered_df['stat_time_day'] < date(2025, 8, 20))]
tt_data_post = tt_data_post.rename(columns={'stat_time_day': 'Date'})
tt_data_post = tt_data_post.loc[tt_data_post['adgroup_id'].isin(tiktok_adgroups),['adgroup_id','Date','impressions','engaged_view_15s','spend']]
tt_data_post = tt_data_post.groupby(by=['Date','adgroup_id']).sum().reset_index()
tt_data_post = tt_data_post[~((tt_data_post['impressions'] == 0) & (tt_data_post['spend'] == 0))].reset_index(drop=True)
tt_data_post.tail()

,Date,adgroup_id,impressions,engaged_view_15s,spend
43,2025-08-18,1839058353004833,31170,2747,897960.000000
44,2025-08-19,1839054793361425,25721,2341,512487.000000
45,2025-08-19,1839054926886321,27493,2124,625936.000000
46,2025-08-19,1839057950747698,52795,5968,1014527
47,2025-08-19,1839058353004833,35049,2923,944236.000000


In [93]:
# Starting from 21st Aug change introduce a stricter spend restriction.

tt_data_post_new = filtered_df.loc[(filtered_df['stat_time_day'] > date(2025, 8, 19)) & (filtered_df['stat_time_day'] < today)]
tt_data_post_new = tt_data_post_new.rename(columns={'stat_time_day': 'Date'})
tt_data_post_new = tt_data_post_new.loc[tt_data_post_new['adgroup_id'].isin(tiktok_adgroups),['adgroup_id','Date','impressions','engaged_view_15s','spend']]
tt_data_post_new = tt_data_post_new.groupby(by=['Date','adgroup_id']).sum().reset_index()
tt_data_post_new = tt_data_post_new[~((tt_data_post_new['impressions'] == 0) & (tt_data_post_new['spend'] == 0))].reset_index(drop=True)
tt_data_post_new.tail()

,Date,adgroup_id,impressions,engaged_view_15s,spend
0,2025-08-20,1839054793361425,23465,2112,503079.000000
1,2025-08-20,1839054926886321,29112,2124,650213.000000
2,2025-08-20,1839057950747698,51417,5413,1001333
3,2025-08-20,1839058353004833,35177,2773,967827.000000


In [94]:
#Convert tt_data_post data types adgroup/date
tt_data_post['adgroup_id'] = tt_data_post['adgroup_id'].astype('int64')
tt_data_post['Date'] = pd.to_datetime(tt_data_post['Date'])

tt_data_post_new['adgroup_id'] = tt_data_post_new['adgroup_id'].astype('int64')
tt_data_post_new['Date'] = pd.to_datetime(tt_data_post_new['Date'])

In [95]:
tt_data_post.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              48 non-null     datetime64[ns]
 1   adgroup_id        48 non-null     int64         
 2   impressions       48 non-null     int64         
 3   engaged_view_15s  48 non-null     int64         
 4   spend             48 non-null     float64       
dtypes: datetime64[ns](1), float64(1), int64(3)
memory usage: 2.0 KB


In [96]:
tt_data_post_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              4 non-null      datetime64[ns]
 1   adgroup_id        4 non-null      int64         
 2   impressions       4 non-null      int64         
 3   engaged_view_15s  4 non-null      int64         
 4   spend             4 non-null      float64       
dtypes: datetime64[ns](1), float64(1), int64(3)
memory usage: 292.0 bytes


In [97]:
# 3. Merge on both adgroup_id and date for tt and historical df
merged_tt = tt_data_post.merge(
historical_df[['adgroup_id', 'date', 'allocated_budget', 'new_budget', 'total_budget']],
left_on=['adgroup_id', 'Date'],
right_on=['adgroup_id', 'date'],
how='left'
)

In [98]:
# 3. Merge on both adgroup_id and date for tt and historical df
merged_tt_new = tt_data_post_new.merge(
historical_df[['adgroup_id', 'date', 'allocated_budget', 'new_budget', 'total_budget']],
left_on=['adgroup_id', 'Date'],
right_on=['adgroup_id', 'date'],
how='left'
)

In [99]:
merged_tt_new.tail()

,Date,adgroup_id,impressions,engaged_view_15s,spend,date,allocated_budget,new_budget,total_budget
0,2025-08-20,1839054793361425,23465,2112,503079.000000,2025-08-20,419640.217569,80094.675805,499734.893374
1,2025-08-20,1839054926886321,29112,2124,650213.000000,2025-08-20,501721.274677,146978.936662,648700.211339
2,2025-08-20,1839057950747698,51417,5413,1001333,2025-08-20,784972.342431,210134.766655,995107.109086
3,2025-08-20,1839058353004833,35177,2773,967827.000000,2025-08-20,702815.535323,261752.764020,964568.299343


In [100]:
merged_tt.tail()

,Date,adgroup_id,impressions,engaged_view_15s,spend,date,allocated_budget,new_budget,total_budget
43,2025-08-18,1839058353004833,31170,2747,897960.000000,2025-08-18,700827.632960,193969.595780,894797.228740
44,2025-08-19,1839054793361425,25721,2341,512487.000000,2025-08-19,421387.293064,88008.442843,509395.735907
45,2025-08-19,1839054926886321,27493,2124,625936.000000,2025-08-19,504860.470741,119127.086246,623987.556987
46,2025-08-19,1839057950747698,52795,5968,1014527,2025-08-19,783482.986936,220451.030142,1003934
47,2025-08-19,1839058353004833,35049,2923,944236.000000,2025-08-19,699798.479259,239667.160662,939465.639921


In [72]:
# Apply and create new columns
merged_tt[['base_spend', 'float_spend', 'rule_applied']] = merged_tt.apply(split_budget, axis=1)

# Final output with impressions & completed views
tt_data_post_complete = merged_tt[['Date','adgroup_id', 'impressions', 'engaged_view_15s', 'spend', 'total_budget', 'base_spend', 'float_spend', 'rule_applied']]

tt_data_post_complete

,Date,adgroup_id,impressions,engaged_view_15s,spend,total_budget,base_spend,float_spend,rule_applied
0,2025-08-08,1839054793361425,28650,3485,739507.000000,733852.500000,607767.377970,131739.722801,Proportional
1,2025-08-08,1839054926886321,28037,2891,977879.000000,976127.000000,608147.076192,369731.923808,Proportional
2,2025-08-08,1839057950747698,49503,8025,1138203,1126917,609836.311324,528366.486673,Proportional
3,2025-08-08,1839058353004833,28858,2999,1121431,1117665,600327.826483,521103.374191,Proportional
4,2025-08-09,1839054793361425,31631,3292,647861.000000,647861.600000,538749.601051,109111.398949,Proportional
5,2025-08-09,1839054926886321,35852,2981,982561.000000,982561.500000,583675.302983,398885.797017,Proportional
6,2025-08-09,1839057950747698,65520,9304,1192328,1192328,667957.700000,524370.400000,Proportional
7,2025-08-09,1839058353004833,34851,3229,1141863,1141863,621635.100000,520228.300000,Proportional
8,2025-08-10,1839054793361425,26938,2952,583364.000000,578289.249000,499114.802877,84249.197154,Proportional
9,2025-08-10,1839054926886321,37790,2678,886805.000000,881593.689700,577560.020126,309244.979874,Proportional


In [101]:
# Apply and create new columns
merged_tt_new[['base_spend', 'float_spend', 'rule_applied']] = merged_tt_new.apply(split_budget_new, axis=1)

# Final output with impressions & completed views
tt_data_post_complete_new = merged_tt_new[['Date','adgroup_id', 'impressions', 'engaged_view_15s', 'spend', 'total_budget', 'base_spend', 'float_spend', 'rule_applied']]

tt_data_post_complete_new

,Date,adgroup_id,impressions,engaged_view_15s,spend,total_budget,base_spend,float_spend,rule_applied
0,2025-08-20,1839054793361425,23465,2112,503079.000000,499734.893374,422448.349742,80630.650258,Proportional
1,2025-08-20,1839054926886321,29112,2124,650213.000000,648700.211339,502891.303979,147321.696021,Proportional
2,2025-08-20,1839057950747698,51417,5413,1001333,995107.109086,789883.524483,211449.475517,Proportional
3,2025-08-20,1839058353004833,35177,2773,967827.000000,964568.299343,705189.929596,262637.070404,Proportional


In [102]:
tt_data_post_complete_final = pd.concat([tt_data_post_complete,tt_data_post_complete_new], ignore_index=True)
tt_data_post_complete_final


,Date,adgroup_id,impressions,engaged_view_15s,spend,total_budget,base_spend,float_spend,rule_applied
0,2025-08-08,1839054793361425,28650,3485,739507.000000,733852.500000,607767.377970,131739.722801,Proportional
1,2025-08-08,1839054926886321,28037,2891,977879.000000,976127.000000,608147.076192,369731.923808,Proportional
2,2025-08-08,1839057950747698,49503,8025,1138203,1126917,609836.311324,528366.486673,Proportional
3,2025-08-08,1839058353004833,28858,2999,1121431,1117665,600327.826483,521103.374191,Proportional
4,2025-08-09,1839054793361425,31631,3292,647861.000000,647861.600000,538749.601051,109111.398949,Proportional
5,2025-08-09,1839054926886321,35852,2981,982561.000000,982561.500000,583675.302983,398885.797017,Proportional
6,2025-08-09,1839057950747698,65520,9304,1192328,1192328,667957.700000,524370.400000,Proportional
7,2025-08-09,1839058353004833,34851,3229,1141863,1141863,621635.100000,520228.300000,Proportional
8,2025-08-10,1839054793361425,26938,2952,583364.000000,578289.249000,499114.802877,84249.197154,Proportional
9,2025-08-10,1839054926886321,37790,2678,886805.000000,881593.689700,577560.020126,309244.979874,Proportional


In [103]:
# Combine the learning phrase and post tt data

tt_data_post_complete_renamed = tt_data_post_complete_final[
['Date', 'adgroup_id', 'impressions', 'spend', 'engaged_view_15s', 'base_spend', 'float_spend']
]

tt_data['Date'] = pd.to_datetime(tt_data['Date'])
tt_data_post_complete_renamed['Date'] = pd.to_datetime(tt_data_post_complete_renamed['Date'])

# Step 4: Append to dv_data
tt_data_combined = pd.concat([tt_data, tt_data_post_complete_renamed], ignore_index=True)

# Optional: Sort by Date then adgroup_id
tt_data_combined = tt_data_combined.sort_values(by=['Date', 'adgroup_id']).reset_index(drop=True)
tt_data_combined.tail()

/tmp/ipython-input-786259048.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tt_data_post_complete_renamed['Date'] = pd.to_datetime(tt_data_post_complete_renamed['Date'])


,Date,adgroup_id,impressions,engaged_view_15s,spend,base_spend,float_spend
79,2025-08-19,1839058353004833,35049,2923,944236.000000,703351.872366,240884.127634
80,2025-08-20,1839054793361425,23465,2112,503079.000000,422448.349742,80630.650258
81,2025-08-20,1839054926886321,29112,2124,650213.000000,502891.303979,147321.696021
82,2025-08-20,1839057950747698,51417,5413,1001333,789883.524483,211449.475517
83,2025-08-20,1839058353004833,35177,2773,967827.000000,705189.929596,262637.070404


#### DV360 split

In [104]:
# DV360 before the 1st budget change

dv_data = report_df.loc[(report_df['Date'] >= Start_date) & (report_df['Date'] < date(2025, 8, 8))]
dv_data = dv_data.loc[dv_data['Line Item ID'].isin(dv_adgroups_1),['Line Item ID','Date','Impressions','Revenue (Adv Currency)','Complete Views (Video)']]
dv_data = dv_data.rename(columns={'Line Item ID': 'adgroup_id', 'Impressions': 'impressions','Revenue (Adv Currency)': 'spend',  'Complete Views (Video)': 'completed_views'})
dv_data = dv_data.groupby(by=['Date','adgroup_id']).sum().reset_index()
dv_data = dv_data[~((dv_data['impressions'] == 0) & (dv_data['spend'] == 0))].reset_index(drop=True)
dv_data['base_spend'] = dv_data['spend'] * 0.75
dv_data['float_spend'] = dv_data['spend'] * 0.25
dv_data.tail()

,Date,adgroup_id,impressions,spend,completed_views,base_spend,float_spend
43,2025-08-07,22853603024,10628,228974.340406,8836,171730.755304,57243.585102
44,2025-08-07,22871714300,15220,525409.456437,13631,394057.092328,131352.364109
45,2025-08-07,22871714861,28647,520015.855278,25500,390011.891459,130003.963820
46,2025-08-07,22875518767,7503,286690.469249,6577,215017.851937,71672.617312
47,2025-08-07,22875518857,13914,517436.267472,12861,388077.200604,129359.066868


In [105]:
# using a new logic for after day 1 of budget allocation which is based how much was spend yesterday.

dv_data_post = report_df.loc[(report_df['Date'] > date(2025, 8, 7)) & (report_df['Date'] < date(2025, 8, 20))]
dv_data_post = dv_data_post.loc[dv_data_post['Line Item ID'].isin(dv_adgroups_1),['Line Item ID','Date','Impressions','Revenue (Adv Currency)','Complete Views (Video)']]
dv_data_post = dv_data_post.rename(columns={'Line Item ID': 'adgroup_id', 'Impressions': 'impressions','Revenue (Adv Currency)': 'spend',  'Complete Views (Video)': 'completed_views'})
dv_data_post = dv_data_post.groupby(by=['Date','adgroup_id']).sum().reset_index()
dv_data_post = dv_data_post[~((dv_data_post['impressions'] == 0) & (dv_data_post['spend'] == 0))].reset_index(drop=True)
dv_data_post.tail()

,Date,adgroup_id,impressions,spend,completed_views
79,2025-08-19,22853603024,2769,58273.096361,2183
80,2025-08-19,22871714300,21268,790912.429736,18738
81,2025-08-19,22871714861,32253,746606.675686,27824
82,2025-08-19,22875518767,2964,110224.650502,2590
83,2025-08-19,22875518857,14073,515756.668553,12989


In [106]:
# using a new logic for after day 1 of budget allocation which is based how much was spend yesterday.

dv_data_post_new = report_df.loc[(report_df['Date'] > date(2025, 8, 19)) & (report_df['Date'] < today)]
dv_data_post_new = dv_data_post_new.loc[dv_data_post_new['Line Item ID'].isin(dv_adgroups_1),['Line Item ID','Date','Impressions','Revenue (Adv Currency)','Complete Views (Video)']]
dv_data_post_new = dv_data_post_new.rename(columns={'Line Item ID': 'adgroup_id', 'Impressions': 'impressions','Revenue (Adv Currency)': 'spend',  'Complete Views (Video)': 'completed_views'})
dv_data_post_new = dv_data_post_new.groupby(by=['Date','adgroup_id']).sum().reset_index()
dv_data_post_new = dv_data_post_new[~((dv_data_post_new['impressions'] == 0) & (dv_data_post_new['spend'] == 0))].reset_index(drop=True)
dv_data_post_new.tail()

,Date,adgroup_id,impressions,spend,completed_views
2,2025-08-20,22853603024,2137,45626.217761,1719
3,2025-08-20,22871714300,22674,823191.092759,20136
4,2025-08-20,22871714861,33406,779290.925486,28564
5,2025-08-20,22875518767,3805,142409.778464,3315
6,2025-08-20,22875518857,13002,481824.230206,11995


In [107]:
# aligned on dv360 adgroup to int
dv_data_post['adgroup_id'] = dv_data_post['adgroup_id'].astype('Int64')
dv_data_post['Date'] = pd.to_datetime(dv_data_post['Date'])

# aligned on dv360 adgroup to int
dv_data_post_new['adgroup_id'] = dv_data_post_new['adgroup_id'].astype('Int64')
dv_data_post_new['Date'] = pd.to_datetime(dv_data_post_new['Date'])

# Merge spend with budget info
# merged = dv_data_post.merge(historical_df[['adgroup_id', 'allocated_budget', 'new_budget', 'total_budget']],on='adgroup_id',how='left')

merged = dv_data_post.merge(
historical_df[['adgroup_id', 'date', 'allocated_budget', 'new_budget', 'total_budget']],
left_on=['adgroup_id', 'Date'],
right_on=['adgroup_id', 'date'],
how='left'
)


merged_new = dv_data_post_new.merge(
historical_df[['adgroup_id', 'date', 'allocated_budget', 'new_budget', 'total_budget']],
left_on=['adgroup_id', 'Date'],
right_on=['adgroup_id', 'date'],
how='left'
)



In [108]:
# Apply and create new columns
merged[['base_spend', 'float_spend', 'rule_applied']] = merged.apply(split_budget, axis=1)

# Final output with impressions & completed views
dv_data_post_complete = merged[['Date','adgroup_id', 'impressions', 'completed_views', 'spend', 'total_budget', 'base_spend', 'float_spend', 'rule_applied']]

dv_data_post_complete

,Date,adgroup_id,impressions,completed_views,spend,total_budget,base_spend,float_spend,rule_applied
0,2025-08-08,22852881341,21847,19306,316444.076421,456523.800000,237333.057316,79111.019105,Fixed 75/25
1,2025-08-08,22853602793,51358,41032,419334.922518,420913.100000,377446.869255,41888.013413,Proportional
2,2025-08-08,22853603024,10957,9173,214802.450243,287931.700000,161101.837682,53700.612561,Fixed 75/25
3,2025-08-08,22871714300,15258,13746,552331.247813,552254.800000,486589.348519,65741.939300,Proportional
4,2025-08-08,22871714861,29527,26569,546448.752749,547500.900000,481237.212817,65211.450105,Proportional
...,...,...,...,...,...,...,...,...,...
79,2025-08-19,22853603024,2769,2183,58273.096361,135396.101154,43704.822271,14568.274090,Fixed 75/25
80,2025-08-19,22871714300,21268,18738,790912.429736,790912.378978,543692.750966,247219.678770,Proportional
81,2025-08-19,22871714861,32253,27824,746606.675686,746786.962512,522382.492951,224224.182735,Proportional
82,2025-08-19,22875518767,2964,2590,110224.650502,192129.539760,82668.487877,27556.162626,Fixed 75/25


In [109]:
# Apply and create new columns
merged_new[['base_spend', 'float_spend', 'rule_applied']] = merged_new.apply(split_budget_new, axis=1)

# Final output with impressions & completed views
dv_data_post_complete_new = merged_new[['Date','adgroup_id', 'impressions', 'completed_views', 'spend', 'total_budget', 'base_spend', 'float_spend', 'rule_applied']]

dv_data_post_complete_new

,Date,adgroup_id,impressions,completed_views,spend,total_budget,base_spend,float_spend,rule_applied
0,2025-08-20,22852881341,13306,10629,309649.790625,736759.129352,232237.342969,77412.447656,Fixed 75/25
1,2025-08-20,22853602793,33131,24023,642024.141141,642018.365311,474739.218204,167284.922937,Proportional
2,2025-08-20,22853603024,2137,1719,45626.217761,127397.335392,34219.663321,11406.554440,Fixed 75/25
3,2025-08-20,22871714300,22674,20136,823191.092759,836697.479384,544152.189242,279038.903517,Proportional
4,2025-08-20,22871714861,33406,28564,779290.925486,782386.218266,526528.479736,252762.445750,Proportional
5,2025-08-20,22875518767,3805,3315,142409.778464,170951.057194,106807.333848,35602.444616,Fixed 75/25
6,2025-08-20,22875518857,13002,11995,481824.230206,590269.401958,361368.172654,120456.057551,Fixed 75/25


In [110]:
dv_data_post_complete_final = pd.concat([dv_data_post_complete,dv_data_post_complete_new], ignore_index=True)
dv_data_post_complete_final

,Date,adgroup_id,impressions,completed_views,spend,total_budget,base_spend,float_spend,rule_applied
0,2025-08-08,22852881341,21847,19306,316444.076421,456523.800000,237333.057316,79111.019105,Fixed 75/25
1,2025-08-08,22853602793,51358,41032,419334.922518,420913.100000,377446.869255,41888.013413,Proportional
2,2025-08-08,22853603024,10957,9173,214802.450243,287931.700000,161101.837682,53700.612561,Fixed 75/25
3,2025-08-08,22871714300,15258,13746,552331.247813,552254.800000,486589.348519,65741.939300,Proportional
4,2025-08-08,22871714861,29527,26569,546448.752749,547500.900000,481237.212817,65211.450105,Proportional
...,...,...,...,...,...,...,...,...,...
86,2025-08-20,22853603024,2137,1719,45626.217761,127397.335392,34219.663321,11406.554440,Fixed 75/25
87,2025-08-20,22871714300,22674,20136,823191.092759,836697.479384,544152.189242,279038.903517,Proportional
88,2025-08-20,22871714861,33406,28564,779290.925486,782386.218266,526528.479736,252762.445750,Proportional
89,2025-08-20,22875518767,3805,3315,142409.778464,170951.057194,106807.333848,35602.444616,Fixed 75/25


In [116]:
test = dv_data_post_complete_final[dv_data_post_complete_final['Date'] =='2025-08-20']
test = test[test['adgroup_id'].isin(fixed_ids)]
test

,Date,adgroup_id,impressions,completed_views,spend,total_budget,base_spend,float_spend,rule_applied
84,2025-08-20,22852881341,13306,10629,309649.790625,736759.129352,232237.342969,77412.447656,Fixed 75/25
86,2025-08-20,22853603024,2137,1719,45626.217761,127397.335392,34219.663321,11406.554440,Fixed 75/25
89,2025-08-20,22875518767,3805,3315,142409.778464,170951.057194,106807.333848,35602.444616,Fixed 75/25
90,2025-08-20,22875518857,13002,11995,481824.230206,590269.401958,361368.172654,120456.057551,Fixed 75/25


In [132]:
# Ensure Date column is datetime
dv_data_post_complete_final['Date'] = pd.to_datetime(dv_data_post_complete_final['Date'])

# Step 1: Find the last 3 unique dates
last_three_dates = sorted(dv_data_post_complete_final['Date'].unique())[-2:]

# Step 2: Filter for only those last 3 dates
df_last_three = dv_data_post_complete_final[dv_data_post_complete_final['Date'].isin(last_three_dates)]

# Step 3: Identify adgroup_ids with Fixed 75/25 for all 2 days
fixed_ids = (
df_last_three.groupby('adgroup_id')['rule_applied']
.apply(lambda x: (x == "Fixed 75/25").all())
)
fixed_ids = fixed_ids[fixed_ids].index.tolist()

# Step 4: Filter again for only those IDs
df_fixed = df_last_three[df_last_three['adgroup_id'].isin(fixed_ids)]

# Step 5: Calculate gap and gap percentage
df_fixed['gap'] = df_fixed['total_budget'] - df_fixed['spend']
df_fixed['gap_pct'] = (df_fixed['gap'] / df_fixed['total_budget']) * 100

# Step 6a: Average gap and gap percentage over the last 2 days
avg_gap_df = (
df_fixed.groupby('adgroup_id')[['spend','total_budget','gap', 'gap_pct']]
.mean()
.reset_index()
.rename(columns={
    'gap': 'avg_gap_last_2_days',
    'gap_pct': 'avg_gap_pct_last_2_days',
    'spend': 'avg_spend_last_2_days',
    'total_budget': 'avg_total_budget_last_2_days'
})
)



# Step 6b: Per-day gap and percentage (optional)
per_day_gap_df = df_fixed[['Date', 'adgroup_id','spend','total_budget', 'gap', 'gap_pct']].sort_values(['adgroup_id', 'Date'])



# Step 7: Calculate average base_spend and float_spend
avg_spend_df  = (
 df_fixed.groupby('adgroup_id')[['base_spend', 'float_spend']]
 .mean()
 .reset_index()
 .rename(columns={
     'base_spend': 'avg_base_spend',
     'float_spend': 'avg_float_spend'
 })
)

print("Average gap over last 2 days (absolute and %):")
display(avg_gap_df)

print("\nGap per day for each ID (absolute and %):")
display(per_day_gap_df)

print("Adgroup IDs with Fixed 75/25 for last 3 days:", fixed_ids)
display(avg_spend_df)

Average gap over last 2 days (absolute and %):


/tmp/ipython-input-2628674755.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fixed['gap'] = df_fixed['total_budget'] - df_fixed['spend']
/tmp/ipython-input-2628674755.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fixed['gap_pct'] = (df_fixed['gap'] / df_fixed['total_budget']) * 100


,adgroup_id,avg_spend_last_2_days,avg_total_budget_last_2_days,avg_gap_last_2_days,avg_gap_pct_last_2_days
0,22852881341,347228.546562,772057.346078,424828.799516,55.154343
1,22853603024,51949.657061,131396.718273,79447.061212,60.573458
2,22875518767,126317.214483,181540.298477,55223.083994,29.662810
3,22875518857,498790.449380,608801.660736,110011.211356,18.079046



Gap per day for each ID (absolute and %):


,Date,adgroup_id,spend,total_budget,gap,gap_pct
77,2025-08-19,22852881341,384807.302499,807355.562804,422548.260305,52.337319
84,2025-08-20,22852881341,309649.790625,736759.129352,427109.338727,57.971367
79,2025-08-19,22853603024,58273.096361,135396.101154,77123.004793,56.961023
86,2025-08-20,22853603024,45626.217761,127397.335392,81771.117631,64.185893
82,2025-08-19,22875518767,110224.650502,192129.539760,81904.889258,42.630035
89,2025-08-20,22875518767,142409.778464,170951.057194,28541.278730,16.695585
83,2025-08-19,22875518857,515756.668553,627333.919513,111577.250960,17.785943
90,2025-08-20,22875518857,481824.230206,590269.401958,108445.171752,18.372149


Adgroup IDs with Fixed 75/25 for last 3 days: [22852881341, 22853603024, 22875518767, 22875518857]


,adgroup_id,avg_base_spend,avg_float_spend
0,22852881341,260421.409922,86807.136641
1,22853603024,38962.242796,12987.414265
2,22875518767,94737.910862,31579.303621
3,22875518857,374092.837035,124697.612345


In [111]:
dv_data_post_complete_renamed = dv_data_post_complete_final[['Date', 'adgroup_id', 'impressions', 'spend', 'completed_views', 'base_spend', 'float_spend']]

dv_data['Date'] = pd.to_datetime(dv_data['Date'])
# dv_data_post_complete_renamed['Date'] = pd.to_datetime(tt_data_post_complete_renamed['Date'])

# Step 4: Append to dv_data
dv_data_combined = pd.concat([dv_data, dv_data_post_complete_renamed], ignore_index=True)

# Optional: Sort by Date then adgroup_id
dv_data_combined = dv_data_combined.sort_values(by=['Date', 'adgroup_id']).reset_index(drop=True)
dv_data_combined

,Date,adgroup_id,impressions,spend,completed_views,base_spend,float_spend
0,2025-07-31,22852881341,8824,169601.906020,7858,127201.429515,42400.476505
1,2025-07-31,22853602793,7178,130771.478932,5820,98078.609199,32692.869733
2,2025-07-31,22853603024,648,10978.052099,553,8233.539074,2744.513025
3,2025-07-31,22871714300,6257,200442.960507,5763,150332.220380,50110.740127
4,2025-07-31,22871714861,6933,200046.519743,6206,150034.889807,50011.629936
...,...,...,...,...,...,...,...
134,2025-08-20,22853603024,2137,45626.217761,1719,34219.663321,11406.554440
135,2025-08-20,22871714300,22674,823191.092759,20136,544152.189242,279038.903517
136,2025-08-20,22871714861,33406,779290.925486,28564,526528.479736,252762.445750
137,2025-08-20,22875518767,3805,142409.778464,3315,106807.333848,35602.444616


### Determine the base spend's pacing amount for each line item for each platform

In [133]:
#Define the pacing function 1st

def daily_base_budget_spend(df,end_date,total_budget):

    #number of days left.
    today = dt.date.today()
    days_left = (end_date - today).days + 1

    #budget left for each day

    total_daily_budget = (total_budget-df['base_spend'].sum()) / days_left

    return round(total_daily_budget,2)



#### DV360 YouTube base allocation

In [167]:
#DV360 YouTube base allocation
df_youtube_filtered = dv_data_combined.loc[dv_data_combined['adgroup_id'].isin(dv_adgroup_id_yt)]
df_youtube_filtered.head()

,Date,adgroup_id,impressions,spend,completed_views,base_spend,float_spend
3,2025-07-31,22871714300,6257,200442.960507,5763,150332.220380,50110.740127
4,2025-07-31,22871714861,6933,200046.519743,6206,150034.889807,50011.629936
5,2025-07-31,22875518767,3000,126833.314322,2510,95124.985742,31708.328581
6,2025-07-31,22875518857,3859,200563.732861,3492,150422.799646,50140.933215
10,2025-08-01,22871714300,6730,200166.362681,5895,150124.772011,50041.590670


In [168]:
Daily_base_budget_yt = daily_base_budget_spend(df_youtube_filtered,End_date,dv_base_yt_budget)
Daily_base_budget_yt

np.float64(1607950.57)

In [144]:
# import pandas as pd

# # Example decay rate
# decay_rate = 0.2

# # Ensure Date is datetime
# df_youtube_filtered['Date'] = pd.to_datetime(df_youtube_filtered['Date'])

# # Find the most recent date
# max_date = df_youtube_filtered['Date'].max()

# # Days ago from most recent date
# df_youtube_filtered['days_ago'] = (max_date - df_youtube_filtered['Date']).dt.days

# # -------------------------
# # 1. Normal spend share
# # -------------------------
# total_spend = df_youtube_filtered['spend'].sum()
# normal_spend_share = (
#  df_youtube_filtered.groupby('adgroup_id')['spend'].sum() / total_spend
# )

# # -------------------------
# # 2. Control decay rate method
# # -------------------------
# df_youtube_filtered['recency_weight_decay'] = decay_rate ** df_youtube_filtered['days_ago']
# df_youtube_filtered['weighted_spend_decay'] = (
#  df_youtube_filtered['spend'] * df_youtube_filtered['recency_weight_decay']
# )
# total_weighted_spend_decay = df_youtube_filtered['weighted_spend_decay'].sum()
# weighted_spend_share_decay = (
#  df_youtube_filtered.groupby('adgroup_id')['weighted_spend_decay'].sum() / total_weighted_spend_decay
# )

# # -------------------------
# # 3. Inverse days method
# # -------------------------
# df_youtube_filtered['recency_weight_inverse'] = 1 / (df_youtube_filtered['days_ago'] + 1)
# df_youtube_filtered['weighted_spend_inverse'] = (
#  df_youtube_filtered['spend'] * df_youtube_filtered['recency_weight_inverse']
# )
# total_weighted_spend_inverse = df_youtube_filtered['weighted_spend_inverse'].sum()
# weighted_spend_share_inverse = (
#  df_youtube_filtered.groupby('adgroup_id')['weighted_spend_inverse'].sum() / total_weighted_spend_inverse
# )

# # -------------------------
# # 4. Combine into comparison DataFrame
# # -------------------------
# comparison_df = pd.DataFrame({
#  'normal_spend_share': normal_spend_share,
#  f'weighted_spend_share_decay_{decay_rate}': weighted_spend_share_decay,
#  'weighted_spend_share_inverse': weighted_spend_share_inverse
# }).reset_index()

# # Optional: Add percentage change from normal
# comparison_df[f'change_pct_decay_{decay_rate}'] = (
#  (comparison_df[f'weighted_spend_share_decay_{decay_rate}'] - comparison_df['normal_spend_share'])
#  / comparison_df['normal_spend_share'] * 100
# )

# comparison_df['change_pct_inverse'] = (
#  (comparison_df['weighted_spend_share_inverse'] - comparison_df['normal_spend_share'])
#  / comparison_df['normal_spend_share'] * 100
# )

# display(comparison_df)

In [179]:
# --- Step 0: Filter avg_spend_df to only YouTube IDs ---
youtube_ids = df_youtube_filtered['adgroup_id'].unique()
avg_spend_df_youtube = avg_spend_df[avg_spend_df['adgroup_id'].isin(youtube_ids)]

# If empty, fixed_allocations will be an empty dict
if avg_spend_df_youtube.empty:
 fixed_allocations = {}
else:
 fixed_allocations = dict(zip(avg_spend_df_youtube['adgroup_id'], avg_spend_df_youtube['avg_base_spend']))

# Total reserved budget for fixed IDs
reserved_budget = sum(fixed_allocations.values())

# Remaining budget for non-fixed IDs
remaining_budget = Daily_base_budget_yt - reserved_budget
if remaining_budget < 0:
 raise ValueError("Fixed allocations exceed total daily budget!")

# --- Step 1: Recency weights ---
df_youtube_filtered['Date'] = pd.to_datetime(df_youtube_filtered['Date'])
max_date = df_youtube_filtered['Date'].max()
df_youtube_filtered['days_ago'] = (max_date - df_youtube_filtered['Date']).dt.days
df_youtube_filtered['recency_weight'] = 1 / (df_youtube_filtered['days_ago'] + 1)
df_youtube_filtered['weighted_spend'] = df_youtube_filtered['spend'] * df_youtube_filtered['recency_weight']

# Weighted spend per ID
weighted_spend_per_id = df_youtube_filtered.groupby('adgroup_id')['weighted_spend'].sum()

# --- Step 2: Allocate remaining budget proportionally to non-fixed IDs ---
non_fixed_ids = weighted_spend_per_id.drop(index=fixed_allocations.keys(), errors='ignore')
total_weight_non_fixed = non_fixed_ids.sum()

if total_weight_non_fixed > 0:
 proportions_non_fixed = non_fixed_ids / total_weight_non_fixed
 non_fixed_allocations = proportions_non_fixed * remaining_budget
else:
 non_fixed_allocations = pd.Series(dtype=float)  # empty if no non-fixed IDs

# --- Step 3: Combine allocations ---
final_allocations = pd.concat([
 pd.Series(fixed_allocations),
 non_fixed_allocations
])
final_allocations = final_allocations.sort_index()

# --- Step 4: Create final allocation DataFrame ---
yt_allocated__base_budget = final_allocations.reset_index()
yt_allocated__base_budget.columns = ['adgroup_id', 'allocated_base_budget']

# Merge with avg_spend_df_youtube (will be empty if no fixed IDs)
yt_allocated__base_budget = yt_allocated__base_budget.merge(avg_spend_df_youtube, on='adgroup_id', how='left')

# Add allocation method column
yt_allocated__base_budget['allocation_method'] = yt_allocated__base_budget['adgroup_id'].apply(
 lambda x: 'fixed_avg_base_spend' if x in fixed_allocations else 'recency_weighted'
)

yt_allocated__base_budget

/tmp/ipython-input-4079316474.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_youtube_filtered['Date'] = pd.to_datetime(df_youtube_filtered['Date'])
/tmp/ipython-input-4079316474.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_youtube_filtered['days_ago'] = (max_date - df_youtube_filtered['Date']).dt.days
/tmp/ipython-input-4079316474.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See

,adgroup_id,allocated_base_budget,avg_base_spend,avg_float_spend,allocation_method
0,22871714300,582879.344574,NaN,NaN,recency_weighted
1,22871714861,556240.477529,NaN,NaN,recency_weighted
2,22875518767,94737.910862,94737.910862,31579.303621,fixed_avg_base_spend
3,22875518857,374092.837035,374092.837035,124697.612345,fixed_avg_base_spend


In [180]:
yt_allocated__base_budget['allocated_base_budget'].sum()

np.float64(1607950.57)

In [181]:
yt_allocated__base_budget = yt_allocated__base_budget.rename(columns={'allocated_base_budget': 'allocated_budget'})
yt_allocated__base_budget = yt_allocated__base_budget[['adgroup_id','allocated_budget']]
yt_allocated__base_budget

,adgroup_id,allocated_budget
0,22871714300,582879.344574
1,22871714861,556240.477529
2,22875518767,94737.910862
3,22875518857,374092.837035


In [172]:
# df_youtube_filtered['Date'] = pd.to_datetime(df_youtube_filtered['Date'])
# max_date = df_youtube_filtered['Date'].max()
# df_youtube_filtered['days_ago'] = (max_date - df_youtube_filtered['Date']).dt.days

# # Apply recency weights (more recent = higher weight)
# df_youtube_filtered['recency_weight'] = 1 / (df_youtube_filtered['days_ago'] + 1)
# df_youtube_filtered['weighted_spend'] = df_youtube_filtered['spend'] * df_youtube_filtered['recency_weight']

# # Calculate averages and proportions
# avg_base_spend = df_youtube_filtered.groupby('adgroup_id')['base_spend'].mean()
# weighted_total = df_youtube_filtered['weighted_spend'].sum()
# spend_proportions = df_youtube_filtered.groupby('adgroup_id')['weighted_spend'].sum() / weighted_total


# yt_allocated__base_budget = spend_proportions * Daily_base_budget_yt

In [146]:
# yt_allocated__base_budget

,weighted_spend
adgroup_id,
22871714300,561717.018915
22871714861,536045.316660
22875518767,124091.211584
22875518857,386097.022841


#### DV360 Video Base allocation

In [147]:
#DV360 Video base allocation

df_video_filtered = dv_data_combined.loc[dv_data_combined['adgroup_id'].isin(dv_adgroup_id_video)]
df_video_filtered.head()

,Date,adgroup_id,impressions,spend,completed_views,base_spend,float_spend
0,2025-07-31,22852881341,8824,169601.906020,7858,127201.429515,42400.476505
1,2025-07-31,22853602793,7178,130771.478932,5820,98078.609199,32692.869733
2,2025-07-31,22853603024,648,10978.052099,553,8233.539074,2744.513025
7,2025-08-01,22852881341,12119,200165.049758,10654,150123.787318,50041.262440
8,2025-08-01,22853602793,13908,200026.185297,11281,150019.638973,50006.546324


In [148]:
Daily_base_budget_video = daily_base_budget_spend(df_video_filtered,End_date,dv_base_video_budget)
Daily_base_budget_video

np.float64(1055630.15)

In [183]:
# --- Step 0: Filter avg_spend_df to only YouTube IDs ---
video_ids = df_video_filtered['adgroup_id'].unique()
avg_spend_df_video = avg_spend_df[avg_spend_df['adgroup_id'].isin(video_ids)]

# If empty, fixed_allocations will be an empty dict
if avg_spend_df_video.empty:
 fixed_allocations = {}
else:
 fixed_allocations = dict(zip(avg_spend_df_video['adgroup_id'], avg_spend_df_video['avg_base_spend']))

# Total reserved budget for fixed IDs
reserved_budget = sum(fixed_allocations.values())

# Remaining budget for non-fixed IDs
remaining_budget = Daily_base_budget_video - reserved_budget
if remaining_budget < 0:
 raise ValueError("Fixed allocations exceed total daily budget!")

# --- Step 1: Recency weights ---
df_video_filtered['Date'] = pd.to_datetime(df_video_filtered['Date'])
max_date = df_video_filtered['Date'].max()
df_video_filtered['days_ago'] = (max_date - df_video_filtered['Date']).dt.days
df_video_filtered['recency_weight'] = 1 / (df_video_filtered['days_ago'] + 1)
df_video_filtered['weighted_spend'] = df_video_filtered['spend'] * df_video_filtered['recency_weight']

# Weighted spend per ID
weighted_spend_per_id = df_video_filtered.groupby('adgroup_id')['weighted_spend'].sum()

# --- Step 2: Allocate remaining budget proportionally to non-fixed IDs ---
non_fixed_ids = weighted_spend_per_id.drop(index=fixed_allocations.keys(), errors='ignore')
total_weight_non_fixed = non_fixed_ids.sum()

if total_weight_non_fixed > 0:
 proportions_non_fixed = non_fixed_ids / total_weight_non_fixed
 non_fixed_allocations = proportions_non_fixed * remaining_budget
else:
 non_fixed_allocations = pd.Series(dtype=float)  # empty if no non-fixed IDs

# --- Step 3: Combine allocations ---
final_allocations = pd.concat([
 pd.Series(fixed_allocations),
 non_fixed_allocations
])
final_allocations = final_allocations.sort_index()

# --- Step 4: Create final allocation DataFrame ---
video_allocated__base_budget = final_allocations.reset_index()
video_allocated__base_budget.columns = ['adgroup_id', 'allocated_base_budget']

# Merge with avg_spend_df_youtube (will be empty if no fixed IDs)
video_allocated__base_budget = video_allocated__base_budget.merge(avg_spend_df_youtube, on='adgroup_id', how='left')

# Add allocation method column
video_allocated__base_budget['allocation_method'] = video_allocated__base_budget['adgroup_id'].apply(
 lambda x: 'fixed_avg_base_spend' if x in fixed_allocations else 'recency_weighted'
)

video_allocated__base_budget

/tmp/ipython-input-125433620.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_video_filtered['Date'] = pd.to_datetime(df_video_filtered['Date'])
/tmp/ipython-input-125433620.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_video_filtered['days_ago'] = (max_date - df_video_filtered['Date']).dt.days
/tmp/ipython-input-125433620.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveat

,adgroup_id,allocated_base_budget,avg_base_spend,avg_float_spend,allocation_method
0,22852881341,260421.409922,NaN,NaN,fixed_avg_base_spend
1,22853602793,756246.497283,NaN,NaN,recency_weighted
2,22853603024,38962.242796,NaN,NaN,fixed_avg_base_spend


In [184]:
video_allocated__base_budget['allocated_base_budget'].sum()

np.float64(1055630.15)

In [185]:
video_allocated__base_budget = video_allocated__base_budget.rename(columns={'allocated_base_budget': 'allocated_budget'})
video_allocated__base_budget = video_allocated__base_budget[['adgroup_id','allocated_budget']]
video_allocated__base_budget

,adgroup_id,allocated_budget
0,22852881341,260421.409922
1,22853602793,756246.497283
2,22853603024,38962.242796


In [178]:
# df_video_filtered['Date'] = pd.to_datetime(df_video_filtered['Date'])
# max_date = df_video_filtered['Date'].max()
# df_video_filtered['days_ago'] = (max_date - df_video_filtered['Date']).dt.days

# # Apply recency weights (more recent = higher weight)
# df_video_filtered['recency_weight'] = 1 / (df_video_filtered['days_ago'] + 1)
# df_video_filtered['weighted_spend'] = df_video_filtered['spend'] * df_video_filtered['recency_weight']

# # Calculate averages and proportions
# avg_base_spend = df_video_filtered.groupby('adgroup_id')['base_spend'].mean()
# weighted_total = df_video_filtered['weighted_spend'].sum()
# spend_proportions = df_video_filtered.groupby('adgroup_id')['weighted_spend'].sum() / weighted_total


# video_allocated__base_budget = spend_proportions * Daily_base_budget_video

In [150]:
# video_allocated__base_budget

,weighted_spend
adgroup_id,
22852881341,444392.651442
22853602793,517672.269860
22853603024,93565.228698


#### TikTok NU CLASS base allocation


In [186]:
df_tiktok_nuclass_filtered = tt_data_combined.loc[tt_data_combined['adgroup_id'].isin(tiktok_adgroup_nuclass)]
df_tiktok_nuclass_filtered.head()

,Date,adgroup_id,impressions,engaged_view_15s,spend,base_spend,float_spend
1,2025-07-31,1839054926886321,9773,794,250000.000000,175000.000000,75000.000000
3,2025-07-31,1839058353004833,7262,613,218069.000000,152648.300000,65420.700000
5,2025-08-01,1839054926886321,8293,1204,244090.000000,170863.000000,73227.000000
7,2025-08-01,1839058353004833,6500,906,192306.000000,134614.200000,57691.800000
9,2025-08-02,1839054926886321,10989,1361,248461.000000,173922.700000,74538.300000


In [187]:
Daily_base_budget_nuclass = daily_base_budget_spend(df_tiktok_nuclass_filtered,End_date,total_base_budget_tt/2)
Daily_base_budget_nuclass

np.float64(1204450.36)

In [188]:
df_tiktok_nuclass_filtered['Date'] = pd.to_datetime(df_tiktok_nuclass_filtered['Date'])
max_date = df_tiktok_nuclass_filtered['Date'].max()
df_tiktok_nuclass_filtered['days_ago'] = (max_date - df_tiktok_nuclass_filtered['Date']).dt.days

# Apply recency weights (more recent = higher weight)
df_tiktok_nuclass_filtered['recency_weight'] = 1 / (df_tiktok_nuclass_filtered['days_ago'] + 1)
df_tiktok_nuclass_filtered['weighted_spend'] = df_tiktok_nuclass_filtered['spend'] * df_tiktok_nuclass_filtered['recency_weight']

# Calculate averages and proportions
avg_base_spend = df_tiktok_nuclass_filtered.groupby('adgroup_id')['base_spend'].mean()
weighted_total = df_tiktok_nuclass_filtered['weighted_spend'].sum()
spend_proportions = df_tiktok_nuclass_filtered.groupby('adgroup_id')['weighted_spend'].sum() / weighted_total


nuclass_allocated__base_budget = spend_proportions * Daily_base_budget_nuclass

/tmp/ipython-input-3497991271.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tiktok_nuclass_filtered['Date'] = pd.to_datetime(df_tiktok_nuclass_filtered['Date'])
/tmp/ipython-input-3497991271.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tiktok_nuclass_filtered['days_ago'] = (max_date - df_tiktok_nuclass_filtered['Date']).dt.days
/tmp/ipython-input-3497991271.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

In [189]:
nuclass_allocated__base_budget

,weighted_spend
adgroup_id,
1839054926886321,500500.216041
1839058353004833,703950.143959


#### TikTok KOL CLASS base allocation

In [190]:
df_tiktok_kol_filtered = tt_data_combined.loc[tt_data_combined['adgroup_id'].isin(tiktok_adgroup_kol)]
df_tiktok_kol_filtered.head()

,Date,adgroup_id,impressions,engaged_view_15s,spend,base_spend,float_spend
0,2025-07-31,1839054793361425,16843,1096,211188.000000,147831.600000,63356.400000
2,2025-07-31,1839057950747698,15657,2027,212000.000000,148400.000000,63600.000000
4,2025-08-01,1839054793361425,10042,2077,196898.000000,137828.600000,59069.400000
6,2025-08-01,1839057950747698,11197,3277,197236.000000,138065.200000,59170.800000
8,2025-08-02,1839054793361425,12257,1994,191914.000000,134339.800000,57574.200000


In [191]:
Daily_base_budget_kol = daily_base_budget_spend(df_tiktok_kol_filtered,End_date,total_base_budget_tt/2)
Daily_base_budget_kol

np.float64(1204424.28)

In [192]:
df_tiktok_kol_filtered['Date'] = pd.to_datetime(df_tiktok_kol_filtered['Date'])
max_date = df_tiktok_kol_filtered['Date'].max()
df_tiktok_kol_filtered['days_ago'] = (max_date - df_tiktok_kol_filtered['Date']).dt.days

# Apply recency weights (more recent = higher weight)
df_tiktok_kol_filtered['recency_weight'] = 1 / (df_tiktok_kol_filtered['days_ago'] + 1)
df_tiktok_kol_filtered['weighted_spend'] = df_tiktok_kol_filtered['spend'] * df_tiktok_kol_filtered['recency_weight']

# Calculate averages and proportions
avg_base_spend = df_tiktok_kol_filtered.groupby('adgroup_id')['base_spend'].mean()
weighted_total = df_tiktok_kol_filtered['weighted_spend'].sum()
spend_proportions = df_tiktok_kol_filtered.groupby('adgroup_id')['weighted_spend'].sum() / weighted_total


kol_allocated__base_budget = spend_proportions * Daily_base_budget_kol

/tmp/ipython-input-2880221910.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tiktok_kol_filtered['Date'] = pd.to_datetime(df_tiktok_kol_filtered['Date'])
/tmp/ipython-input-2880221910.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tiktok_kol_filtered['days_ago'] = (max_date - df_tiktok_kol_filtered['Date']).dt.days
/tmp/ipython-input-2880221910.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value ins

In [193]:
kol_allocated__base_budget

,weighted_spend
adgroup_id,
1839054793361425,417958.606651
1839057950747698,786465.673349


#### Combining into one dataframe base

In [195]:
df1 = nuclass_allocated__base_budget.reset_index()
df1.columns = ['adgroup_id', 'allocated_budget']
df2 = kol_allocated__base_budget.reset_index()
df2.columns = ['adgroup_id', 'allocated_budget']


In [196]:
combined__base_df = pd.concat([df1, df2,video_allocated__base_budget,yt_allocated__base_budget], ignore_index=True)
combined__base_df


,adgroup_id,allocated_budget
0,1839054926886321,500500.216041
1,1839058353004833,703950.143959
2,1839054793361425,417958.606651
3,1839057950747698,786465.673349
4,22852881341,260421.409922
5,22853602793,756246.497283
6,22853603024,38962.242796
7,22871714300,582879.344574
8,22871714861,556240.477529
9,22875518767,94737.910862


### Floating budget and performanc calculation

In [197]:
# Combined the splited Tiktok and DV360 data
combined_data = pd.concat([tt_data_combined, dv_data_combined], ignore_index=True).fillna(0)
combined_data.tail(6)

,Date,adgroup_id,impressions,engaged_view_15s,spend,base_spend,float_spend,completed_views
217,2025-08-20,22853602793,33131,0.000000,642024.141141,474739.218204,167284.922937,24023.000000
218,2025-08-20,22853603024,2137,0.000000,45626.217761,34219.663321,11406.554440,1719.000000
219,2025-08-20,22871714300,22674,0.000000,823191.092759,544152.189242,279038.903517,20136.000000
220,2025-08-20,22871714861,33406,0.000000,779290.925486,526528.479736,252762.445750,28564.000000
221,2025-08-20,22875518767,3805,0.000000,142409.778464,106807.333848,35602.444616,3315.000000
222,2025-08-20,22875518857,13002,0.000000,481824.230206,361368.172654,120456.057551,11995.000000


In [198]:
combined_data['float_spend'].sum()

np.float64(29438420.88217239)

#### Delivery capacity calcuation to be revised*

In [199]:
# # Number of days for initial delivery capacity

n = 8

In [200]:
learning_start = max(tt_data['Date'].min(), dv_data['Date'].min())
cutoff_date = learning_start + dt.timedelta(days=n-1)
combined_common_data = combined_data.loc[combined_data['Date'] >= learning_start,:]
learning_phase_df = combined_common_data.loc[combined_common_data['Date'] <= cutoff_date,:]
# learning_phase_max = learning_phase_df[['adgroup_id','Date','spend']].groupby(by=['adgroup_id','Date']).sum().reset_index()
# learning_phase_max = learning_phase_max[['adgroup_id','spend']].groupby(by=['adgroup_id']).max().reset_index()

In [201]:
combined_common_data

,Date,adgroup_id,impressions,engaged_view_15s,spend,base_spend,float_spend,completed_views
0,2025-07-31,1839054793361425,16843,1096.000000,211188.000000,147831.600000,63356.400000,0.000000
1,2025-07-31,1839054926886321,9773,794.000000,250000.000000,175000.000000,75000.000000,0.000000
2,2025-07-31,1839057950747698,15657,2027.000000,212000.000000,148400.000000,63600.000000,0.000000
3,2025-07-31,1839058353004833,7262,613.000000,218069.000000,152648.300000,65420.700000,0.000000
4,2025-08-01,1839054793361425,10042,2077.000000,196898.000000,137828.600000,59069.400000,0.000000
...,...,...,...,...,...,...,...,...
218,2025-08-20,22853603024,2137,0.000000,45626.217761,34219.663321,11406.554440,1719.000000
219,2025-08-20,22871714300,22674,0.000000,823191.092759,544152.189242,279038.903517,20136.000000
220,2025-08-20,22871714861,33406,0.000000,779290.925486,526528.479736,252762.445750,28564.000000
221,2025-08-20,22875518767,3805,0.000000,142409.778464,106807.333848,35602.444616,3315.000000


In [202]:
# # Delivery capacity for first day of algo

# today = dt.date.today()
# # if(cutoff_date + dt.timedelta(days=1) == today):
# #     budget_cap = learning_phase_max.copy()
# budget_cap = learning_phase_max.copy()
# if(budget_cap['spend'].sum()) < daily_overall_target:
#     gap = daily_overall_target/budget_cap['spend'].sum() + 1
#     budget_cap['spend'] = budget_cap['spend'] * gap

#### Calulate daily desired delivery

In [203]:
def daily_float_budget_spend(df,end_date,total_budget):

    #number of days left.
    today = dt.date.today()
    days_left = (end_date - today).days + 1

    #budget left for each day

    total_daily_budget = (total_budget-df['float_spend'].sum()) / days_left

    return round(total_daily_budget,2)

In [204]:
daily_overall_target = daily_float_budget_spend(combined_data,End_date,Total_budget)

In [205]:
daily_overall_target

np.float64(1938087.3)

#### Each child performance calculation

In [206]:
def calculate_cpm(dataframe, alpha):
    # Get a list of unique ad set in the dataframe
    adgroups = dataframe['adgroup_id'].unique()

    # Initialize an empty dictionary to store the results
    cpm_dict = {}

    # Loop through each ad set
    for adgroup in adgroups:
        # Filter the dataframe by ad set and sort by date in descending order #add reset index
        filtered_df = dataframe[dataframe['adgroup_id'] == adgroup].sort_values(by='Date', ascending=False)

        # Initialize variables for total cost and total clicks
        total_cost = 0
        total_imps = 0

        # Loop through each row in the filtered dataframe using enumerate()
        for i, (_, row) in enumerate(filtered_df.iterrows()):
            # Get the cost and click values for this row
            cost = row['spend']
            impressions = row['impressions']

            # Calculate the weight for this row based on its position in the dataframe
            weight = (1 - alpha) ** i

            # Add the weighted cost and clicks to the totals
            total_cost += weight * cost
            total_imps += weight * impressions

        # Calculate the cost per click using the weighted totals
        if total_imps != 0:
            cpm = total_cost / (total_imps / 1000)
        else:
            cpm = 0 #to extremely hit number

        # Add the result to the dictionary
        cpm_dict[adgroup] =cpm

    return cpm_dict

In [217]:
def calculate_cpcv(dataframe, alpha):
    # Get a list of unique ad set in the dataframe
    adgroups = dataframe['adgroup_id'].unique()

    # Initialize an empty dictionary to store the results
    cpcv_dict = {}

    # Loop through each ad set
    for adgroup in adgroups:
        # Filter the dataframe by ad set and sort by date in descending order #add reset index
        filtered_df = dataframe[dataframe['adgroup_id'] == adgroup].sort_values(by='Date', ascending=False)

        # Initialize variables for total cost and total clicks
        total_cost = 0
        total_completed_views = 0

        # Loop through each row in the filtered dataframe using enumerate()
        for i, (_, row) in enumerate(filtered_df.iterrows()):
            # Get the cost and click values for this row
            cost = row['spend']
            completed_views = row['completed_views']

            # Calculate the weight for this row based on its position in the dataframe
            weight = (1 - alpha) ** i

            # Add the weighted cost and clicks to the totals
            total_cost += weight * cost
            total_completed_views += weight * completed_views

        # Calculate the cost per click using the weighted totals
        if total_completed_views != 0:
            cpcv = total_cost / total_completed_views
        else:
            cpcv = 0 #to extremely hit number

        # Add the result to the dictionary
        cpcv_dict[adgroup] =cpcv

    return cpcv_dict

In [207]:
def calculate_cpv(dataframe, alpha):
    # Get a list of unique ad set in the dataframe
    adgroups = dataframe['adgroup_id'].unique()

    # Initialize an empty dictionary to store the results
    cpv_dict = {}

    # Loop through each ad set
    for adgroup in adgroups:
        # Filter the dataframe by ad set and sort by date in descending order #add reset index
        filtered_df = dataframe[dataframe['adgroup_id'] == adgroup].sort_values(by='Date', ascending=False)

        # Initialize variables for total cost and total clicks
        total_cost = 0
        total_engaged_view = 0

        # Loop through each row in the filtered dataframe using enumerate()
        for i, (_, row) in enumerate(filtered_df.iterrows()):
            # Get the cost and click values for this row
            cost = row['spend']
            engaged_view = row['engaged_view_15s']

            # Calculate the weight for this row based on its position in the dataframe
            weight = (1 - alpha) ** i

            # Add the weighted cost and clicks to the totals
            total_cost += weight * cost
            total_engaged_view += weight * engaged_view

        # Calculate the cost per click using the weighted totals
        if total_engaged_view != 0:
            cpv = total_cost / total_engaged_view
        else:
            cpv = 0 #to extremely hit number

        # Add the result to the dictionary
        cpv_dict[adgroup] =cpv

    return cpv_dict

In [208]:
def calculate_fvr(dataframe, alpha):
    # Get a list of unique ad set in the dataframe
    adgroups = dataframe['adgroup_id'].unique()

    # Initialize an empty dictionary to store the results
    fvr_dict = {}

    # Loop through each ad set
    for adgroup in adgroups:
        # Filter the dataframe by ad set and sort by date in descending order #add reset index
        filtered_df = dataframe[dataframe['adgroup_id'] == adgroup].sort_values(by='Date', ascending=False)

        # Initialize variables for total cost and total clicks
        total_imps = 0
        total_engaged_view_15s = 0

        # Loop through each row in the filtered dataframe using enumerate()
        for i, (_, row) in enumerate(filtered_df.iterrows()):
            # Get the cost and click values for this row
            impressions = row['impressions']
            engaged_view_15s = row['engaged_view_15s']

            # Calculate the weight for this row based on its position in the dataframe
            weight = (1 - alpha) ** i

            # Add the weighted cost and clicks to the totals
            total_imps += weight * impressions
            total_engaged_view_15s += weight * engaged_view_15s

        # Calculate the cost per click using the weighted totals
        if total_imps != 0:
            fvr = total_engaged_view_15s / total_imps
        else:
            fvr = 0 #to extremely hit number

        # Add the result to the dictionary
        fvr_dict[adgroup] =fvr

    return fvr_dict

In [209]:
def calculate_vcr(dataframe, alpha):
    # Get a list of unique ad set in the dataframe
    adgroups = dataframe['adgroup_id'].unique()

    # Initialize an empty dictionary to store the results
    vcr_dict = {}

    # Loop through each ad set
    for adgroup in adgroups:
        # Filter the dataframe by ad set and sort by date in descending order #add reset index
        filtered_df = dataframe[dataframe['adgroup_id'] == adgroup].sort_values(by='Date', ascending=False)

        # Initialize variables for total cost and total clicks
        total_imps = 0
        total_video_completes = 0

        # Loop through each row in the filtered dataframe using enumerate()
        for i, (_, row) in enumerate(filtered_df.iterrows()):
            # Get the cost and click values for this row
            impressions = row['impressions']
            video_complete = row['completed_views']

            # Calculate the weight for this row based on its position in the dataframe
            weight = (1 - alpha) ** i

            # Add the weighted cost and clicks to the totals
            total_imps += weight * impressions
            total_video_completes += weight * video_complete

        # Calculate the cost per click using the weighted totals
        if total_imps != 0:
            vcr = total_video_completes / total_imps
        else:
            vcr = 0 #to extremely hit number

        # Add the result to the dictionary
        vcr_dict[adgroup] =vcr

    return vcr_dict

In [218]:
cpm_performance = calculate_cpm(combined_data, 0.5)
vcr_performance = calculate_vcr(combined_data, 0.5)
cpv_performance = calculate_cpv(combined_data,0.5)
fvr_performance = calculate_fvr(combined_data, 0.5)
cpcv_performance = calculate_cpcv(combined_data,0.5)

In [219]:
cpcv_performance

{np.float64(1839054793361425.0): 0,
 np.float64(1839054926886321.0): 0,
 np.float64(1839057950747698.0): 0,
 np.float64(1839058353004833.0): 0,
 np.float64(22852881341.0): 26.53844064678,
 np.float64(22853602793.0): 21.714208389982975,
 np.float64(22853603024.0): 25.158806911245232,
 np.float64(22871714300.0): 41.34605500654029,
 np.float64(22871714861.0): 26.457760638072692,
 np.float64(22875518767.0): 43.14099141013234,
 np.float64(22875518857.0): 40.155095328115664}

In [220]:
cpm_performance

{np.float64(1839054793361425.0): 21020.209082231326,
 np.float64(1839054926886321.0): 22398.281834993428,
 np.float64(1839057950747698.0): 19498.815024270552,
 np.float64(1839058353004833.0): 27492.414607875748,
 np.float64(22852881341.0): 22026.79048066498,
 np.float64(22853602793.0): 16166.215965469086,
 np.float64(22853603024.0): 20388.70987365534,
 np.float64(22871714300.0): 36625.8724279061,
 np.float64(22871714861.0): 22775.23317729915,
 np.float64(22875518767.0): 37372.40473541964,
 np.float64(22875518857.0): 37001.13394951338}

In [221]:
vcr_performance

{np.float64(1839054793361425.0): 0.0,
 np.float64(1839054926886321.0): 0.0,
 np.float64(1839057950747698.0): 0.0,
 np.float64(1839058353004833.0): 0.0,
 np.float64(22852881341.0): 0.8299956570107507,
 np.float64(22853602793.0): 0.7444994390367349,
 np.float64(22853603024.0): 0.8104005068913741,
 np.float64(22871714300.0): 0.8858371716990282,
 np.float64(22871714861.0): 0.8608148470632699,
 np.float64(22875518767.0): 0.8662852547853627,
 np.float64(22875518857.0): 0.9214555126110245}

In [222]:
cpv_performance

{np.float64(1839054793361425.0): 226.49379573316756,
 np.float64(1839054926886321.0): 301.4235753674688,
 np.float64(1839057950747698.0): 175.23196302660324,
 np.float64(1839058353004833.0): 337.0631122834791,
 np.float64(22852881341.0): 0,
 np.float64(22853602793.0): 0,
 np.float64(22853603024.0): 0,
 np.float64(22871714300.0): 0,
 np.float64(22871714861.0): 0,
 np.float64(22875518767.0): 0,
 np.float64(22875518857.0): 0}

In [223]:
fvr_performance

{np.float64(1839054793361425.0): 0.09280699726978502,
 np.float64(1839054926886321.0): 0.07430832776662355,
 np.float64(1839057950747698.0): 0.11127430571162575,
 np.float64(1839058353004833.0): 0.081564590149378,
 np.float64(22852881341.0): 0.0,
 np.float64(22853602793.0): 0.0,
 np.float64(22853603024.0): 0.0,
 np.float64(22871714300.0): 0.0,
 np.float64(22871714861.0): 0.0,
 np.float64(22875518767.0): 0.0,
 np.float64(22875518857.0): 0.0}

#### Comparing againts benchmarks

In [224]:
benchmark_reference_df = combined_common_data.loc[combined_common_data['Date'] <= cutoff_date,:].copy()

benchmark_reference_df['platform'] = 'Unknown'

# Update platform based on adgroup_id
benchmark_reference_df.loc[benchmark_reference_df['adgroup_id'].isin(dv_adgroup_id_video), 'platform'] = 'DV360_video'
benchmark_reference_df.loc[benchmark_reference_df['adgroup_id'].isin(dv_adgroup_id_yt), 'platform'] = 'DV360_yt'
benchmark_reference_df.loc[benchmark_reference_df['adgroup_id'].isin(tiktok_adgroup_kol), 'platform'] = 'TikTok_kol'
benchmark_reference_df.loc[benchmark_reference_df['adgroup_id'].isin(tiktok_adgroup_nuclass), 'platform'] = 'TikTok_nuclass'



In [225]:
benchmark_reference_df

,Date,adgroup_id,impressions,engaged_view_15s,spend,base_spend,float_spend,completed_views,platform
0,2025-07-31,1839054793361425,16843,1096.000000,211188.000000,147831.600000,63356.400000,0.000000,TikTok_kol
1,2025-07-31,1839054926886321,9773,794.000000,250000.000000,175000.000000,75000.000000,0.000000,TikTok_nuclass
2,2025-07-31,1839057950747698,15657,2027.000000,212000.000000,148400.000000,63600.000000,0.000000,TikTok_kol
3,2025-07-31,1839058353004833,7262,613.000000,218069.000000,152648.300000,65420.700000,0.000000,TikTok_nuclass
4,2025-08-01,1839054793361425,10042,2077.000000,196898.000000,137828.600000,59069.400000,0.000000,TikTok_kol
...,...,...,...,...,...,...,...,...,...
127,2025-08-07,22853603024,10628,0.000000,228974.340406,171730.755304,57243.585102,8836.000000,DV360_video
128,2025-08-07,22871714300,15220,0.000000,525409.456437,394057.092328,131352.364109,13631.000000,DV360_yt
129,2025-08-07,22871714861,28647,0.000000,520015.855278,390011.891459,130003.963820,25500.000000,DV360_yt
130,2025-08-07,22875518767,7503,0.000000,286690.469249,215017.851937,71672.617312,6577.000000,DV360_yt


In [226]:
# prompt: group benchmark_reference_df by adgroup_id. do not use date column while aggregating

# Group benchmark_reference_df by adgroup_id, excluding the 'Date' column
benchmark_reference_grouped = benchmark_reference_df.groupby('platform').agg(
    {'impressions': 'sum', 'spend': 'sum', 'engaged_view_15s': 'sum', 'completed_views': 'sum'}
).reset_index()

# Calculate VCR for benchmark_reference_grouped
benchmark_reference_grouped['VCR_video'] = 0.83
benchmark_reference_grouped['VCR_yt'] = 0.88


# Calculate CPM for benchmark_reference_grouped
benchmark_reference_grouped['CPM_video'] = 40
benchmark_reference_grouped['CPM_yt'] = 50

# Calculate Cost per engage views for benchmark_reference_grouped
benchmark_reference_grouped['fvr_kol'] = 0.11
benchmark_reference_grouped['fvr_nuclass'] = 0.07


benchmark_reference_grouped

,platform,impressions,spend,engaged_view_15s,completed_views,VCR_video,VCR_yt,CPM_video,CPM_yt,fvr_kol,fvr_nuclass
0,DV360_video,337796,4406001,0.000000,290008.000000,0.830000,0.880000,40,50,0.110000,0.070000
1,DV360_yt,194309,5593363,0.000000,173037.000000,0.830000,0.880000,40,50,0.110000,0.070000
2,TikTok_kol,343703,5895548,54518.000000,0.000000,0.830000,0.880000,40,50,0.110000,0.070000
3,TikTok_nuclass,220032,6014368,23968.000000,0.000000,0.830000,0.880000,40,50,0.110000,0.070000


In [227]:
# Filter for active adgroups only

# three_days_ago = today - pd.Timedelta(days=3)
# last_three_days_df = combined_data[combined_data['Date'] >= three_days_ago]
# active_li = last_three_days_df['adgroup_id'].unique()

# active_li
combined_data['Date'] = pd.to_datetime(combined_data['Date'])

# Ensure 'today' is a Pandas Timestamp
today = pd.Timestamp.today().normalize()

# Calculate threshold as Timestamp
three_days_ago = today - pd.Timedelta(days=3)

# Filter
last_three_days_df = combined_data[combined_data['Date'] >= three_days_ago]

# Get unique adgroup IDs
active_li = last_three_days_df['adgroup_id'].unique()

print(active_li)


<FloatingArray>
[1839054793361425.0, 1839054926886321.0, 1839057950747698.0,
 1839058353004833.0,      22852881341.0,      22853602793.0,
      22853603024.0,      22871714300.0,      22871714861.0,
      22875518767.0,      22875518857.0]
Length: 11, dtype: Float64


In [228]:
# Create a new DataFrame from vcr_performance and cpm_performance
new_df = pd.DataFrame({'adgroup_id': list(vcr_performance.keys()),
                       'vcr_performance': list(vcr_performance.values()),
                       'cpm_performance': list(cpm_performance.values()),
                       'cpv_performance': list(cpv_performance.values()),
                       'fvr_performance': list(fvr_performance.values())})

# Merge with benchmark_reference_grouped based on platform

adgroup_to_platform = {}
for adgroup_id in tiktok_adgroup_kol:
    adgroup_to_platform[adgroup_id] = 'TikTok_kol'
for adgroup_id in tiktok_adgroup_nuclass:
    adgroup_to_platform[adgroup_id] = 'TikTok_nuclass'
for adgroup_id in dv_adgroup_id_video:
    adgroup_to_platform[adgroup_id] = 'DV360_video'
for adgroup_id in dv_adgroup_id_yt:
    adgroup_to_platform[adgroup_id] = 'DV360_yt'

new_df['platform'] = new_df['adgroup_id'].map(adgroup_to_platform)


new_df = pd.merge(new_df, benchmark_reference_grouped[['platform', 'VCR_video', 'VCR_yt','CPM_video','CPM_yt','fvr_kol','fvr_nuclass']], on='platform', how='left')

# # Rename columns
new_df = new_df.rename(columns={'VCR_video': 'VCR_video_benchmark', 'VCR_yt': 'VCR_yt_benchmark','CPM_video': 'CPM_video_benchmark','CPM_yt': 'CPM_yt_benchmark','fvr_kol': 'fvr_kol_benchmark','fvr_nuclass': 'fvr_nuclass_benchmark'})

new_df = new_df[new_df['adgroup_id'].isin(active_li)].reset_index(drop=True)

# Display the resulting DataFrame
new_df


,adgroup_id,vcr_performance,cpm_performance,cpv_performance,fvr_performance,platform,VCR_video_benchmark,VCR_yt_benchmark,CPM_video_benchmark,CPM_yt_benchmark,fvr_kol_benchmark,fvr_nuclass_benchmark
0,1839054793361425,0.000000,21020.209082,226.493796,0.092807,TikTok_kol,0.830000,0.880000,40,50,0.110000,0.070000
1,1839054926886321,0.000000,22398.281835,301.423575,0.074308,TikTok_nuclass,0.830000,0.880000,40,50,0.110000,0.070000
2,1839057950747698,0.000000,19498.815024,175.231963,0.111274,TikTok_kol,0.830000,0.880000,40,50,0.110000,0.070000
3,1839058353004833,0.000000,27492.414608,337.063112,0.081565,TikTok_nuclass,0.830000,0.880000,40,50,0.110000,0.070000
4,22852881341,0.829996,22026.790481,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000
5,22853602793,0.744499,16166.215965,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000
6,22853603024,0.810401,20388.709874,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000
7,22871714300,0.885837,36625.872428,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000
8,22871714861,0.860815,22775.233177,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000
9,22875518767,0.866285,37372.404735,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000


In [229]:
new_df.replace([np.inf, -np.inf], 0, inplace=True)
new_df

,adgroup_id,vcr_performance,cpm_performance,cpv_performance,fvr_performance,platform,VCR_video_benchmark,VCR_yt_benchmark,CPM_video_benchmark,CPM_yt_benchmark,fvr_kol_benchmark,fvr_nuclass_benchmark
0,1839054793361425,0.000000,21020.209082,226.493796,0.092807,TikTok_kol,0.830000,0.880000,40,50,0.110000,0.070000
1,1839054926886321,0.000000,22398.281835,301.423575,0.074308,TikTok_nuclass,0.830000,0.880000,40,50,0.110000,0.070000
2,1839057950747698,0.000000,19498.815024,175.231963,0.111274,TikTok_kol,0.830000,0.880000,40,50,0.110000,0.070000
3,1839058353004833,0.000000,27492.414608,337.063112,0.081565,TikTok_nuclass,0.830000,0.880000,40,50,0.110000,0.070000
4,22852881341,0.829996,22026.790481,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000
5,22853602793,0.744499,16166.215965,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000
6,22853603024,0.810401,20388.709874,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000
7,22871714300,0.885837,36625.872428,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000
8,22871714861,0.860815,22775.233177,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000
9,22875518767,0.866285,37372.404735,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000


In [230]:
# Initialize column with zeros (or NaN if preferred)
new_df['vcr_index'] = 0

# Apply for YouTube group
yt_mask_1 = new_df['adgroup_id'].isin(dv_adgroup_id_yt)
new_df.loc[yt_mask_1, 'vcr_index'] = new_df.loc[yt_mask_1, 'VCR_yt_benchmark']/ new_df.loc[yt_mask_1, 'vcr_performance']

# Apply for Video group
video_mask_1 = new_df['adgroup_id'].isin(dv_adgroup_id_video)
new_df.loc[video_mask_1, 'vcr_index'] = new_df.loc[video_mask_1, 'VCR_video_benchmark']/ new_df.loc[video_mask_1, 'vcr_performance']

# Initialize column with zeros (or NaN if preferred)
new_df['fvr_index'] = 0

# Apply for tiktok kol group
tt_kol_mask = new_df['adgroup_id'].isin(tiktok_adgroup_kol)
new_df.loc[tt_kol_mask, 'fvr_index'] = new_df.loc[tt_kol_mask, 'fvr_kol_benchmark']/ new_df.loc[tt_kol_mask, 'fvr_performance']

# Apply for tiktok nuclass group
tt_nuclass_mask = new_df['adgroup_id'].isin(tiktok_adgroup_nuclass)
new_df.loc[tt_nuclass_mask, 'fvr_index'] = new_df.loc[tt_nuclass_mask, 'fvr_nuclass_benchmark']/ new_df.loc[tt_nuclass_mask, 'fvr_performance']


# Replace NaN values with 0
new_df.fillna(0, inplace=True)

new_df


/tmp/ipython-input-4251889635.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.99341056 1.0222872  1.01583167 0.95501084]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  new_df.loc[yt_mask_1, 'vcr_index'] = new_df.loc[yt_mask_1, 'VCR_yt_benchmark']/ new_df.loc[yt_mask_1, 'vcr_performance']
/tmp/ipython-input-4251889635.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.18525546 0.98854807]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  new_df.loc[tt_kol_mask, 'fvr_index'] = new_df.loc[tt_kol_mask, 'fvr_kol_benchmark']/ new_df.loc[tt_kol_mask, 'fvr_performance']


,adgroup_id,vcr_performance,cpm_performance,cpv_performance,fvr_performance,platform,VCR_video_benchmark,VCR_yt_benchmark,CPM_video_benchmark,CPM_yt_benchmark,fvr_kol_benchmark,fvr_nuclass_benchmark,vcr_index,fvr_index
0,1839054793361425,0.000000,21020.209082,226.493796,0.092807,TikTok_kol,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,1.185255
1,1839054926886321,0.000000,22398.281835,301.423575,0.074308,TikTok_nuclass,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,0.942021
2,1839057950747698,0.000000,19498.815024,175.231963,0.111274,TikTok_kol,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,0.988548
3,1839058353004833,0.000000,27492.414608,337.063112,0.081565,TikTok_nuclass,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,0.858216
4,22852881341,0.829996,22026.790481,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000,1.000005,0.000000
5,22853602793,0.744499,16166.215965,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000,1.114843,0.000000
6,22853603024,0.810401,20388.709874,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000,1.024185,0.000000
7,22871714300,0.885837,36625.872428,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000,0.993411,0.000000
8,22871714861,0.860815,22775.233177,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000,1.022287,0.000000
9,22875518767,0.866285,37372.404735,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000,1.015832,0.000000


In [231]:
# Merging the two different platform index into one column
new_df['combined_index'] = new_df['vcr_index'].where(new_df['vcr_index'] != 0, new_df['fvr_index'])

In [232]:
new_df

,adgroup_id,vcr_performance,cpm_performance,cpv_performance,fvr_performance,platform,VCR_video_benchmark,VCR_yt_benchmark,CPM_video_benchmark,CPM_yt_benchmark,fvr_kol_benchmark,fvr_nuclass_benchmark,vcr_index,fvr_index,combined_index
0,1839054793361425,0.000000,21020.209082,226.493796,0.092807,TikTok_kol,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,1.185255,1.185255
1,1839054926886321,0.000000,22398.281835,301.423575,0.074308,TikTok_nuclass,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,0.942021,0.942021
2,1839057950747698,0.000000,19498.815024,175.231963,0.111274,TikTok_kol,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,0.988548,0.988548
3,1839058353004833,0.000000,27492.414608,337.063112,0.081565,TikTok_nuclass,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,0.858216,0.858216
4,22852881341,0.829996,22026.790481,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000,1.000005,0.000000,1.000005
5,22853602793,0.744499,16166.215965,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000,1.114843,0.000000,1.114843
6,22853603024,0.810401,20388.709874,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000,1.024185,0.000000,1.024185
7,22871714300,0.885837,36625.872428,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000,0.993411,0.000000,0.993411
8,22871714861,0.860815,22775.233177,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000,1.022287,0.000000,1.022287
9,22875518767,0.866285,37372.404735,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000,1.015832,0.000000,1.015832


### Optimization

In [233]:
import pandas as pd

# # Assuming 'combined_common_data' is the DataFrame from the previous code
# # and it has columns 'Date', 'adgroup_id', and 'spend'

# # Get today's date
# today = pd.to_datetime('today').date()

# # Calculate the date 3 days ago, excluding today
# three_days_ago = today - pd.Timedelta(days=3)

# # Filter the DataFrame to include data from the last 3 days (excluding today)
# last_three_days_data = combined_common_data[
#     (combined_common_data["Date"] >= three_days_ago) & (combined_common_data["Date"] < today)
# ]

# # Group by 'adgroup_id' and calculate the average spend
# average_spends = last_three_days_data.groupby("adgroup_id")["float_spend"].mean().reset_index()

# # Create the DataFrame
# average_float_spends_df = pd.DataFrame(average_spends)

# # Display or use the DataFrame
# average_float_spends_df
# Ensure Date column is datetime64[ns]
combined_common_data['Date'] = pd.to_datetime(combined_common_data['Date'])

# Get today's date as a Pandas Timestamp (normalized to midnight)
today = pd.Timestamp.today().normalize()

# Calculate the date 3 days ago
three_days_ago = today - pd.Timedelta(days=3)

# Filter for last 3 days excluding today
last_three_days_data = combined_common_data[
(combined_common_data["Date"] >= three_days_ago) &
(combined_common_data["Date"] < today)
]

# Group by adgroup_id and calculate average float_spend
average_spends = last_three_days_data.groupby("adgroup_id")["float_spend"].mean().reset_index()

# Create DataFrame
average_float_spends_df = pd.DataFrame(average_spends)

print(average_float_spends_df)

         adgroup_id   float_spend
0       22852881341 115928.367076
1       22853602793 171090.739538
2       22853603024  15801.757360
3       22871714300 251351.182363
4       22871714861 233175.904405
5       22875518767  30427.262997
6       22875518857 123881.967049
7  1839054793361425  88643.274081
8  1839054926886321 124298.219138
9  1839057950747698 229180.960055
10 1839058353004833 232725.467766


In [234]:

yesterday = date.today() - timedelta(days=1)
day_before_yesterday = yesterday - timedelta(days=1)

# Filter for spends made yesterday and the day before
filtered_data = combined_common_data[
    combined_common_data["Date"].isin([day_before_yesterday])
]

# Group by 'adgroup_id' and calculate the proportion
adgroup_proportions = (last_three_days_data.groupby("adgroup_id")["float_spend"].mean()).reset_index()
adgroup_proportions = adgroup_proportions.rename(columns={"float_spend": "previous_day_spend"})
adgroup_proportions['spend_guardlines_max'] = adgroup_proportions['previous_day_spend'] * 1.5 # SIMON TO RECOMMEND
adgroup_proportions['spend_guardlines_min'] = adgroup_proportions['previous_day_spend'] * 0.5


adgroup_proportions

/tmp/ipython-input-1345892028.py:6: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  combined_common_data["Date"].isin([day_before_yesterday])


,adgroup_id,previous_day_spend,spend_guardlines_max,spend_guardlines_min
0,22852881341,115928.367076,173892.550615,57964.183538
1,22853602793,171090.739538,256636.109308,85545.369769
2,22853603024,15801.757360,23702.636040,7900.878680
3,22871714300,251351.182363,377026.773545,125675.591182
4,22871714861,233175.904405,349763.856607,116587.952202
5,22875518767,30427.262997,45640.894495,15213.631498
6,22875518857,123881.967049,185822.950574,61940.983525
7,1839054793361425,88643.274081,132964.911121,44321.637040
8,1839054926886321,124298.219138,186447.328707,62149.109569
9,1839057950747698,229180.960055,343771.440083,114590.480028


In [235]:
combined_df = new_df.merge(adgroup_proportions, on='adgroup_id', how='left')
# combined_df = combined_df.rename(columns={"spend": "delivery_cap"})
combined_df = combined_df[combined_df['adgroup_id'].isin(active_li)].reset_index(drop=True)
mask_1 = combined_df['spend_guardlines_min'] == 0
combined_df.loc[mask_1, 'spend_guardlines_min'] = 5000
# Minimum budget for each child campaign - RANDOM # SIMON TO RECOMMEND

# combined_df = pd.merge(new_df, adgroup_proportions, on="adgroup_id", how="left")

# combined_df['new_min'] = combined_df[['min_budget','spend_guardlines_min']].max(axis=1)

In [236]:
combined_df

,adgroup_id,vcr_performance,cpm_performance,cpv_performance,fvr_performance,platform,VCR_video_benchmark,VCR_yt_benchmark,CPM_video_benchmark,CPM_yt_benchmark,fvr_kol_benchmark,fvr_nuclass_benchmark,vcr_index,fvr_index,combined_index,previous_day_spend,spend_guardlines_max,spend_guardlines_min
0,1839054793361425,0.000000,21020.209082,226.493796,0.092807,TikTok_kol,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,1.185255,1.185255,88643.274081,132964.911121,44321.637040
1,1839054926886321,0.000000,22398.281835,301.423575,0.074308,TikTok_nuclass,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,0.942021,0.942021,124298.219138,186447.328707,62149.109569
2,1839057950747698,0.000000,19498.815024,175.231963,0.111274,TikTok_kol,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,0.988548,0.988548,229180.960055,343771.440083,114590.480028
3,1839058353004833,0.000000,27492.414608,337.063112,0.081565,TikTok_nuclass,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,0.858216,0.858216,232725.467766,349088.201649,116362.733883
4,22852881341,0.829996,22026.790481,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000,1.000005,0.000000,1.000005,115928.367076,173892.550615,57964.183538
5,22853602793,0.744499,16166.215965,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000,1.114843,0.000000,1.114843,171090.739538,256636.109308,85545.369769
6,22853603024,0.810401,20388.709874,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000,1.024185,0.000000,1.024185,15801.757360,23702.636040,7900.878680
7,22871714300,0.885837,36625.872428,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000,0.993411,0.000000,0.993411,251351.182363,377026.773545,125675.591182
8,22871714861,0.860815,22775.233177,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000,1.022287,0.000000,1.022287,233175.904405,349763.856607,116587.952202
9,22875518767,0.866285,37372.404735,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000,1.015832,0.000000,1.015832,30427.262997,45640.894495,15213.631498


In [241]:
avg_spend_df.head()

,adgroup_id,avg_base_spend,avg_float_spend
0,22852881341,260421.409922,86807.136641
1,22853603024,38962.242796,12987.414265
2,22875518767,94737.910862,31579.303621
3,22875518857,374092.837035,124697.612345


In [257]:
# If avg_spend_df is empty, skip updates
if avg_spend_df.empty:
  print("⚠️ avg_spend_df is empty — no guardline updates applied.")
  final_updated_df = combined_df.copy()

else:

  # Merge avg_spend_df into final_df so we have avg_float_spend alongside guardlines
  merged_df = combined_df.merge(
    avg_spend_df[['adgroup_id', 'avg_float_spend']],
    on='adgroup_id',
    how='left'
  )

  # Update spend_guardlines_max for IDs in avg_spend_df
  mask_new = merged_df['avg_float_spend'].notna()

  # Calculate proposed new max and min
  proposed_max = 1.1 * merged_df['avg_float_spend']
  proposed_min = 0.5 * merged_df['avg_float_spend']

  # For max → choose the smaller between current and proposed
  merged_df.loc[mask_new, 'spend_guardlines_max'] = pd.DataFrame({
    'current': merged_df.loc[mask_new, 'spend_guardlines_max'],
    'proposed': proposed_max[mask_new]
  }).min(axis=1)

  # For min → choose the larger between current and proposed
  merged_df.loc[mask_new, 'spend_guardlines_min'] = pd.DataFrame({
    'current': merged_df.loc[mask_new, 'spend_guardlines_min'],
    'proposed': proposed_min[mask_new]
  }).max(axis=1)

  # Drop helper column if not needed
  final_updated_df = merged_df.drop(columns=['avg_float_spend'])

final_updated_df

,adgroup_id,vcr_performance,cpm_performance,cpv_performance,fvr_performance,platform,VCR_video_benchmark,VCR_yt_benchmark,CPM_video_benchmark,CPM_yt_benchmark,fvr_kol_benchmark,fvr_nuclass_benchmark,vcr_index,fvr_index,combined_index,previous_day_spend,spend_guardlines_max,spend_guardlines_min,new_budget
0,1839054793361425,0.000000,21020.209082,226.493796,0.092807,TikTok_kol,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,1.185255,1.185255,88643.274081,132964.911121,44321.637040,64377.768759
1,1839054926886321,0.000000,22398.281835,301.423575,0.074308,TikTok_nuclass,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,0.942021,0.942021,124298.219138,186447.328707,62149.109569,186447.328696
2,1839057950747698,0.000000,19498.815024,175.231963,0.111274,TikTok_kol,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,0.988548,0.988548,229180.960055,343771.440083,114590.480028,302396.854581
3,1839058353004833,0.000000,27492.414608,337.063112,0.081565,TikTok_nuclass,0.830000,0.880000,40,50,0.110000,0.070000,0.000000,0.858216,0.858216,232725.467766,349088.201649,116362.733883,349088.201649
4,22852881341,0.829996,22026.790481,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000,1.000005,0.000000,1.000005,115928.367076,95487.850305,57964.183538,173892.550611
5,22853602793,0.744499,16166.215965,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000,1.114843,0.000000,1.114843,171090.739538,256636.109308,85545.369769,102023.748619
6,22853603024,0.810401,20388.709874,0.000000,0.000000,DV360_video,0.830000,0.880000,40,50,0.110000,0.070000,1.024185,0.000000,1.024185,15801.757360,14286.155692,7900.878680,23702.636004
7,22871714300,0.885837,36625.872428,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000,0.993411,0.000000,0.993411,251351.182363,377026.773545,125675.591182,287654.837038
8,22871714861,0.860815,22775.233177,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000,1.022287,0.000000,1.022287,233175.904405,349763.856607,116587.952202,217039.529008
9,22875518767,0.866285,37372.404735,0.000000,0.000000,DV360_yt,0.830000,0.880000,40,50,0.110000,0.070000,1.015832,0.000000,1.015832,30427.262997,34737.233983,15789.651810,45640.894466


In [242]:
def objective_function(budget_ratios, kpi_values, regularization_strength, beta):
    kpi_values_inverted = 1 / kpi_values
    softmax_ratios = np.exp(beta * kpi_values_inverted) / np.sum(np.exp(beta * kpi_values_inverted))
    reg_term = np.sum((budget_ratios / softmax_ratios - 1)**2)
    return -np.sum(budget_ratios / kpi_values_inverted) + regularization_strength * reg_term

In [249]:
# Inputs
parent_daily_budget = daily_overall_target
kpi_values = combined_df['combined_index'].values  # KPI values for each child campaign
# delivery_capacity = combined_df['delivery_cap'].values  # Delivery capacity for each child campaign
min_budget = combined_df['spend_guardlines_min'].values
max_budget = combined_df['spend_guardlines_max'].values  # Maximum budget for each child campaign
regularization_strength = 1  # Penalizes extreme allocations (makes sure the output we get is different from upper and lower bounds)
beta = 7 # closer to 0 = uniform distribution (even budget split)

**High Beta, Low Regularization:** This combination can lead to very concentrated allocations. The high beta makes the softmax ratios highly skewed towards the best-performing channels, and the low regularization_strength allows the optimization to allocate most of the budget to those channels.

**Low Beta, High Regularization:** This combination can lead to very even allocations. The low beta makes the softmax ratios nearly uniform, and the high regularization_strength penalizes any deviations from this uniform allocation.

**High Beta, High Regularization:** This combination can lead to a more balanced allocation, but it can also be difficult to tune. The high beta pushes the allocation towards the best-performing channels, while the high regularization_strength pulls it back towards the softmax ratios. The optimal balance depends on the specific KPI values and the desired level of concentration.

**Low Beta, Low Regularization:** This combination can lead to unpredictable results. The low beta makes the softmax ratios nearly uniform, and the low regularization_strength allows the optimization to allocate budget almost arbitrarily.

In [250]:
# Constraints
constraints = [
    {"type": "eq", "fun": lambda x: np.sum(x) - 1},  # Sum of budget_ratios should be equal to 1
]

bounds = [(min_val / parent_daily_budget, max_val / parent_daily_budget) for min_val, max_val in zip(min_budget, max_budget)]


In [251]:
# Initial guess
x0 = np.ones(len(kpi_values)) / len(kpi_values)

In [252]:
# Solve the optimization problem

res = minimize(objective_function, x0, args=(kpi_values, regularization_strength, beta), bounds=bounds, constraints=constraints, method='SLSQP')

In [253]:
# Calculate the daily budget for each child campaign
daily_budgets = res.x * parent_daily_budget

combined_df['new_budget'] = daily_budgets

In [254]:
final_df  = combined_df[['adgroup_id','combined_index','spend_guardlines_max','spend_guardlines_min','new_budget']]
final_df

,adgroup_id,combined_index,spend_guardlines_max,spend_guardlines_min,new_budget
0,1839054793361425,1.185255,132964.911121,44321.637040,64377.768759
1,1839054926886321,0.942021,186447.328707,62149.109569,186447.328696
2,1839057950747698,0.988548,343771.440083,114590.480028,302396.854581
3,1839058353004833,0.858216,349088.201649,116362.733883,349088.201649
4,22852881341,1.000005,173892.550615,57964.183538,173892.550611
5,22853602793,1.114843,256636.109308,85545.369769,102023.748619
6,22853603024,1.024185,23702.636040,7900.878680,23702.636004
7,22871714300,0.993411,377026.773545,125675.591182,287654.837038
8,22871714861,1.022287,349763.856607,116587.952202,217039.529008
9,22875518767,1.015832,45640.894495,15213.631498,45640.894466


In [255]:
final_df['new_budget'].sum()

np.float64(1938087.2999959597)

In [256]:
daily_overall_target

np.float64(1938087.3)

In [ ]:
# Merge the dataframes on adgroup_id
final_combined_float_base = final_df.merge(combined__base_df, on='adgroup_id', how='outer')

# Create new column with sum of allocated_budget and new_budget
final_combined_float_base['platform_total_budget'] = final_combined_float_base['allocated_budget'].fillna(0) + final_combined_float_base['new_budget'].fillna(0)

final_combined_float_base

,adgroup_id,combined_index,spend_guardlines_max,spend_guardlines_min,new_budget,allocated_budget,platform_total_budget
0,22852881341,0.946313,175910.840976,58636.946992,130355.264311,438545.791166,568901.055477
1,22853602793,1.056349,101628.176050,33876.058683,75024.329880,414544.441132,489568.771012
2,22853603024,1.003044,73352.261037,24450.753679,73352.261037,179642.617701,252994.878738
3,22871714300,0.970760,146336.304631,48778.768210,114059.141111,492658.950818,606718.091929
4,22871714861,0.996622,138954.239603,46318.079868,99723.689543,484094.425277,583818.114820
5,22875518767,1.014583,94969.126793,31656.375598,91234.448312,207039.019288,298273.467600
6,22875518857,0.940127,153951.960369,51317.320123,134954.869718,408839.244617,543794.114334
7,1839054793361425,0.982527,144867.036378,48289.012126,107226.105385,430563.403853,537789.509238
8,1839054926886321,0.827644,402126.971728,134042.323909,278931.722700,535855.817106,814787.539806
9,1839057950747698,0.801978,690767.981787,230255.993929,338618.260226,775410.886147,1114029


In [ ]:
final_combined_float_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adgroup_id             11 non-null     float64
 1   combined_index         11 non-null     float64
 2   spend_guardlines_max   11 non-null     float64
 3   spend_guardlines_min   11 non-null     float64
 4   new_budget             11 non-null     float64
 5   allocated_budget       11 non-null     float64
 6   platform_total_budget  11 non-null     float64
dtypes: float64(7)
memory usage: 748.0 bytes


In [ ]:
# # Add today's date
final_combined_float_base['date'] = date.today()

final_combined_float_base

,adgroup_id,combined_index,spend_guardlines_max,spend_guardlines_min,new_budget,allocated_budget,platform_total_budget,date
0,22852881341,0.946313,175910.840976,58636.946992,130355.264311,438545.791166,568901.055477,2025-08-13
1,22853602793,1.056349,101628.176050,33876.058683,75024.329880,414544.441132,489568.771012,2025-08-13
2,22853603024,1.003044,73352.261037,24450.753679,73352.261037,179642.617701,252994.878738,2025-08-13
3,22871714300,0.970760,146336.304631,48778.768210,114059.141111,492658.950818,606718.091929,2025-08-13
4,22871714861,0.996622,138954.239603,46318.079868,99723.689543,484094.425277,583818.114820,2025-08-13
5,22875518767,1.014583,94969.126793,31656.375598,91234.448312,207039.019288,298273.467600,2025-08-13
6,22875518857,0.940127,153951.960369,51317.320123,134954.869718,408839.244617,543794.114334,2025-08-13
7,1839054793361425,0.982527,144867.036378,48289.012126,107226.105385,430563.403853,537789.509238,2025-08-13
8,1839054926886321,0.827644,402126.971728,134042.323909,278931.722700,535855.817106,814787.539806,2025-08-13
9,1839057950747698,0.801978,690767.981787,230255.993929,338618.260226,775410.886147,1114029,2025-08-13


In [ ]:
# Make a copy so we don't modify the original
new_budget_df = final_combined_float_base.copy()

# Rename if needed
if 'platform_total_budget' in new_budget_df.columns and 'total_budget' in historical_df.columns:
  new_budget_df = new_budget_df.rename(columns={'platform_total_budget': 'total_budget'})


# Add any missing columns from historical_df
for col in historical_df.columns:
  if col not in new_budget_df.columns:
    new_budget_df[col] = np.nan  # or calculate if possible



# Reorder columns to match historical_df
new_budget_df = new_budget_df[historical_df.columns]

# Ensure both have datetime64[ns] for date
historical_df['date'] = pd.to_datetime(historical_df['date'])
new_budget_df['date'] = pd.to_datetime(new_budget_df['date'])


# Append
historical_df_new = pd.concat([historical_df, new_budget_df], ignore_index=True)


# Remove duplicates (keep the latest entry for each adgroup_id + date)
historical_df_new = historical_df_new.drop_duplicates(subset=['adgroup_id', 'date'], keep='last')

# Sort for cleanliness
historical_df_new = historical_df_new.sort_values(by=['date', 'adgroup_id']).reset_index(drop=True)

historical_df_new.tail()

,adgroup_id,combined_index,spend_guardlines_max,spend_guardlines_min,new_budget,allocated_budget,total_budget,date
61,22875518857,0.940127,153951.960369,51317.320123,134954.869718,408839.244617,543794.114334,2025-08-13
62,1839054793361425,0.982527,144867.036378,48289.012126,107226.105385,430563.403853,537789.509238,2025-08-13
63,1839054926886321,0.827644,402126.971728,134042.323909,278931.722700,535855.817106,814787.539806,2025-08-13
64,1839057950747698,0.801978,690767.981787,230255.993929,338618.260226,775410.886147,1114029,2025-08-13
65,1839058353004833,0.764092,686360.629527,228786.876509,461673.577778,669150.242894,1130824,2025-08-13


In [ ]:
#SAVING NEW FILE
# Get today's date in YYYYMMDD format
today_str = datetime.today().strftime("%Y%m%d")

# Build filenames with today's date
parquet_filename = f"historical_adjustments_{today_str}.parquet"
csv_filename = f"historical_adjustments_{today_str}.csv"

# Save Parquet (safe for precision)
historical_df_new.to_parquet(parquet_filename, index=False)

# Save CSV (for sharing, with float precision control)
historical_df_new.to_csv(csv_filename, index=False, float_format="%.15g")

print(f"Saved: {parquet_filename} and {csv_filename}")

Saved: historical_adjustments_20250813.parquet and historical_adjustments_20250813.csv


In [ ]:
# ==== CONFIG ====
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "executor_files/mp-adh-groupm-sg-cea49b7167eb.json"
BUCKET_NAME = "apac-github-test"  # Your GCS bucket name
FOLDER_PATH = "Files_for_vertex_ai/Cross_Video_Zott_historical"  # Folder inside bucket

# ==== INIT GCS CLIENT ====
client = storage.Client()

def upload_to_gcs(local_file_path, bucket_name, blob_path):
  """Uploads a file to GCS."""
  bucket = client.bucket(bucket_name)
  blob = bucket.blob(blob_path)
  blob.upload_from_filename(local_file_path)
  print(f"✅ Uploaded {local_file_path} to gs://{bucket_name}/{blob_path}")

In [ ]:
# Upload to GCS in your specified folder
upload_to_gcs(csv_filename, BUCKET_NAME, f"{FOLDER_PATH}/{csv_filename}")
upload_to_gcs(parquet_filename, BUCKET_NAME, f"{FOLDER_PATH}/{parquet_filename}")

✅ Uploaded historical_adjustments_20250813.csv to gs://apac-github-test/Files_for_vertex_ai/Cross_Video_Zott_historical/historical_adjustments_20250813.csv
✅ Uploaded historical_adjustments_20250813.parquet to gs://apac-github-test/Files_for_vertex_ai/Cross_Video_Zott_historical/historical_adjustments_20250813.parquet


### Update budget changes

#### DV360 CHANGES

In [ ]:
# Filter for adgroup_id in yt_adgroups and select specific columns

filtered_df_video = final_combined_float_base[final_combined_float_base['adgroup_id'].isin(dv_adgroup_id_video)][['adgroup_id', 'platform_total_budget']]
filtered_df_yt = final_combined_float_base[final_combined_float_base['adgroup_id'].isin(dv_adgroup_id_yt)][['adgroup_id', 'platform_total_budget']]

# Converting video budget
filtered_df_video['platform_total_budget'] = filtered_df_video['platform_total_budget'].astype(int)
filtered_df_video['adgroup_id'] = filtered_df_video['adgroup_id'].astype(int)
filtered_df_video = filtered_df_video.astype(str)

# Converting YT budget
filtered_df_yt['platform_total_budget'] = filtered_df_yt['platform_total_budget'].astype(int)
filtered_df_yt['adgroup_id'] = filtered_df_yt['adgroup_id'].astype(int)
filtered_df_yt = filtered_df_yt.astype(str)

In [ ]:
filtered_df_video

,adgroup_id,platform_total_budget
0,22852881341,568901
1,22853602793,489568
2,22853603024,252994


In [ ]:
filtered_df_yt

,adgroup_id,platform_total_budget
3,22871714300,606718
4,22871714861,583818
5,22875518767,298273
6,22875518857,543794


In [ ]:
# Inputs for Video
partner_id_input = "1278635"  # DV360 Partner ID
adv_id_input = "6993273759"
strategy_id_input = "183965"  # Put the Copilot Strategy ID
object_inputs = filtered_df_video['adgroup_id']  # List of Line Item IDs
keys = ["Pacing Amount"]
values = filtered_df_video['platform_total_budget']  # Corresponding values for each Line Item

# Generate changes for each object
changes_sdf = {object_id: dict(zip(keys, [value])) for object_id, value in zip(object_inputs, values)}

# Create JSON body
jsonbody = {
    "strategyId": strategy_id_input,
    "deltas": changes_sdf
}

# Convert to JSON string
json_ready = json.dumps(jsonbody, indent=4)
print(json_ready)

{
    "strategyId": "183965",
    "deltas": {
        "22852881341": {
            "Pacing Amount": "568901"
        },
        "22853602793": {
            "Pacing Amount": "489568"
        },
        "22853603024": {
            "Pacing Amount": "252994"
        }
    }
}


In [ ]:
# Updating through API
headers = {"developerapitoken":"e19026e0-de42-425e-913c-3e0f27baf72f", "content-type": "application/json"}

response = requests.post(url="https://optimization.choreograph.com/developerAPI/v1/SDFUpdate/LineItem", data=json_ready, headers=headers, timeout=600)
print(response.status_code)
print(response.content)
print(response.elapsed)


500
b''
0:00:34.670662


In [ ]:
# Inputs for YouTube
partner_id_input = "1278635"  # DV360 Partner ID
adv_id_input = "6993273759"
strategy_id_input = "183964"  # Put the Copilot Strategy ID
object_inputs = filtered_df_yt['adgroup_id']  # List of Line Item IDs
keys = ["Pacing Amount"]
values = filtered_df_yt['platform_total_budget']  # Corresponding values for each Line Item

# Generate changes for each object
changes_sdf = {object_id: dict(zip(keys, [value])) for object_id, value in zip(object_inputs, values)}

# Create JSON body
jsonbody = {
    "strategyId": strategy_id_input,
    "deltas": changes_sdf
}

# Convert to JSON string
json_ready = json.dumps(jsonbody, indent=4)
print(json_ready)

{
    "strategyId": "183964",
    "deltas": {
        "22871714300": {
            "Pacing Amount": "606718"
        },
        "22871714861": {
            "Pacing Amount": "583818"
        },
        "22875518767": {
            "Pacing Amount": "298273"
        },
        "22875518857": {
            "Pacing Amount": "543794"
        }
    }
}


In [ ]:
# Updating through API
headers = {"developerapitoken":"e19026e0-de42-425e-913c-3e0f27baf72f", "content-type": "application/json"}

response = requests.post(url="https://optimization.choreograph.com/developerAPI/v1/SDFUpdate/LineItem", data=json_ready, headers=headers, timeout=600)
print(response.status_code)
print(response.content)
print(response.elapsed)

500
b''
0:00:34.538864


#### TikTok changes

In [ ]:
PATH = "/open_api/v1.3/adgroup/budget/update/"

def build_url(path, query=""):
    # type: (str, str) -> str
    """
    Build request URL
    :param path: Request path
    :param query: Querystring
    :return: Request URL
    """
    scheme, netloc = "https", "business-api.tiktok.com"
    return urlunparse((scheme, netloc, path, "", query, ""))

def post(json_str):
    # type: (str) -> dict
    """
    Send POST request
    :param json_str: Args in JSON format
    :return: Response in JSON format
    """
    url = build_url(PATH)
    args = json.loads(json_str)
    headers = {
        "Access-Token": ACCESS_TOKEN,
        "Content-Type": "application/json",
    }
    rsp = requests.post(url, headers=headers, json=args)
    return rsp.json()



In [ ]:
#kol

advertiser_id = 7371003546684424193
budget_df = final_combined_float_base.loc[final_combined_float_base['adgroup_id'].isin(tiktok_adgroup_kol),['adgroup_id','platform_total_budget']]
#Converting adgroup to str and budget to 2sf
budget_df['adgroup_id'] = budget_df['adgroup_id'].apply(lambda x: str(int(x)))
budget_df['platform_total_budget'] = budget_df['platform_total_budget'].astype(int)
campaign_id = 1839054793361409
budget_df

,adgroup_id,platform_total_budget
7,1839054793361425,537789
9,1839057950747698,1114029


In [ ]:
all_args = []

for index,row in budget_df.iterrows():
  budget_value = row['platform_total_budget']
  adgroup_id = row['adgroup_id']

  # Construct the payload as a Python dictionary
  my_args = {
      "advertiser_id": str(advertiser_id),
      "budget": [
          {
              "adgroup_id": str(adgroup_id),
              "budget": budget_value
          }
      ]
  }
  all_args.append(my_args)

# Iterate through the list of payloads and make the POST request for each
for args in all_args:
  # Use json.dumps to convert the dictionary to a JSON string
  print(post(json.dumps(args)))

{'code': 0, 'message': 'OK', 'request_id': '20250813104012B478F1F7C55903FBA8B2', 'data': {}}
{'code': 0, 'message': 'OK', 'request_id': '202508131040136AFCF0D53554A6FE5C05', 'data': {}}


In [ ]:
#nuclass

advertiser_id = 7371003546684424193
budget_df = final_combined_float_base.loc[final_combined_float_base['adgroup_id'].isin(tiktok_adgroup_nuclass),['adgroup_id','platform_total_budget']]
#Converting adgroup to str and budget to 2sf
budget_df['adgroup_id'] = budget_df['adgroup_id'].apply(lambda x: str(int(x)))
budget_df['platform_total_budget'] = budget_df['platform_total_budget'].astype(int)
campaign_id = 1839054926886305
budget_df

,adgroup_id,platform_total_budget
8,1839054926886321,814787
10,1839058353004833,1130823


In [ ]:
all_args = []

for index,row in budget_df.iterrows():
  budget_value = row['platform_total_budget']
  adgroup_id = row['adgroup_id']

  # Construct the payload as a Python dictionary
  my_args = {
      "advertiser_id": str(advertiser_id),
      "budget": [
          {
              "adgroup_id": str(adgroup_id),
              "budget": budget_value
          }
      ]
  }
  all_args.append(my_args)

# Iterate through the list of payloads and make the POST request for each
for args in all_args:
  # Use json.dumps to convert the dictionary to a JSON string
  print(post(json.dumps(args)))

{'code': 0, 'message': 'OK', 'request_id': '20250813104014888C46DC3021BF0263AD', 'data': {}}
{'code': 0, 'message': 'OK', 'request_id': '20250813104016CEE94AD0B15812F4CE85', 'data': {}}
